In [1]:
import datetime
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from os import cpu_count
import os
from requests.exceptions import RequestException
import numpy as np
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from ipydatagrid import DataGrid
from IPython.display import display, Markdown
from statsmodels.stats.correlation_tools import cov_nearest

# your analytics helpers (rebalanced_portfolio, RiskAnalysis, get_frontier, ...)
from src.RiskMetrics import *
from src.Rebalancing import *
from src.Metrics import *
from src import GitHub
from keys import *
# all yfinance access -- imported last so nothing above can shadow these names
from src.Yahoo_API import (QUOTE_TYPE_CLASS, search_yahoo, fetch_ticker_details, has_price_history,
                           get_close, get_currencies, get_fx_rates, to_base_currency)

from src.PositionHandler import update_files, describe as describe_update



In [2]:
# =============================================================================
# Multi-asset portfolio app -- prices from yfinance, positions / quantities /
# P&L from Excel files (same layout as your existing Positions / Quantities files).
#
# External helpers this module still expects to be importable in your notebook
# (unchanged from your existing modules):
#   RiskAnalysis, display_scrollable_df, build_constraint, beta_constraint,
#   vol_constraint, rebalanced_portfolio, rebalanced_time_series,
#   rebalanced_dynamic_quantities, rebalanced_book_cost, buy_and_hold,
#   rebalanced_metrics, get_portfolio_risk, get_frontier, get_calendar_graph,
#   get_asset_risk, get_asset_returns, set_symmetric, get_var_contribution,
#   get_ex_ante_vol, get_ex_ante_vol_contribution, get_correlation_contribution,
#   get_idiosyncratic_contribution, first_pca_over_time, get_rebalancing_dates,
#   performance_contribution, drawdown_contribution
# =============================================================================


# Excel sources (Binance-style layouts, fiat currencies). Local paths or
# http(s) URLs both work (pd.read_excel handles both).
#   Current Portfolio.xlsx : columns asset, free, locked, ... ; held units =
#                            free + locked. Cash rows use the fiat code
#                            ('USD', 'EUR', ...), units = amount of cash.
#   Trade History.xlsx     : Date(UTC), Market, Type, Price, Amount, Total, Fee,
#                            Fee Coin. Market = asset + trading currency
#                            ('XLKUSD', 'GOLD.PAEUR'); Price/Total/Fee are in
#                            that currency, Amount in units.
#   Positions.xlsx         : index = dates, columns = tickers (+ cash) + 'Total',
#                            values = market value in BASE_CURRENCY
#   Quantities.xlsx        : index = dates, columns = tickers (+ cash), unitshttps://github.com/niroojane/Risk-Management/raw/refs/heads/main/ETF/Current%20Portfolio2.xlsx
# P&L is rebuilt from the trade history (average-cost method). Cost and P&L are
# kept in each asset's trading currency (the same currency yfinance quotes it
# in) and translated to BASE_CURRENCY with that day's FX rate for display.
DEFAULT_CURRENT_PORTFOLIO_PATH = 'https://github.com/niroojane/Risk-Management/raw/refs/heads/main/ETF/Current%20Portfolio2.xlsx'
DEFAULT_TRADES_PATH = 'https://github.com/niroojane/Risk-Management/raw/refs/heads/main/ETF/Trade%20History2.xlsx'
DEFAULT_POSITIONS_PATH = 'https://github.com/niroojane/Risk-Management/raw/refs/heads/main/ETF/Positions2.xlsx'
DEFAULT_QUANTITIES_PATH = 'https://github.com/niroojane/Risk-Management/raw/refs/heads/main/ETF/Quantities2.xlsx'

git=GitHub(token,repo_owner,repo_name,branch)


In [3]:
# =============================================================================
# Multi-asset portfolio app -- prices from yfinance, positions / quantities /
# P&L from Excel files (same layout as your existing Positions / Quantities files).
#
# External helpers this module still expects to be importable in your notebook
# (unchanged from your existing modules):
#   RiskAnalysis, display_scrollable_df, build_constraint, beta_constraint,
#   vol_constraint, rebalanced_portfolio, rebalanced_time_series,
#   rebalanced_dynamic_quantities, rebalanced_book_cost, buy_and_hold,
#   rebalanced_metrics, get_portfolio_risk, get_frontier, get_calendar_graph,
#   get_asset_risk, get_asset_returns, set_symmetric, get_var_contribution,
#   get_ex_ante_vol, get_ex_ante_vol_contribution, get_correlation_contribution,
#   get_idiosyncratic_contribution, first_pca_over_time, get_rebalancing_dates,
#   performance_contribution, drawdown_contribution
# =============================================================================


# Excel sources (Binance-style layouts, fiat currencies). Local paths or
# http(s) URLs both work (pd.read_excel handles both).
#   Current Portfolio.xlsx : columns asset, free, locked, ... ; held units =
#                            free + locked. Cash rows use the fiat code
#                            ('USD', 'EUR', ...), units = amount of cash.
#   Trade History.xlsx     : Date(UTC), Market, Type, Price, Amount, Total, Fee,
#                            Fee Coin. Market = asset + trading currency
#                            ('XLKUSD', 'GOLD.PAEUR'); Price/Total/Fee are in
#                            that currency, Amount in units.
#   Positions.xlsx         : index = dates, columns = tickers (+ cash) + 'Total',
#                            values = market value in BASE_CURRENCY
#   Quantities.xlsx        : index = dates, columns = tickers (+ cash), units
# P&L is rebuilt from the trade history (average-cost method). Cost and P&L are
# kept in each asset's trading currency (the same currency yfinance quotes it
# in) and translated to BASE_CURRENCY with that day's FX rate for display.


# Currency everything is reported in. Also selectable in the Investment
# Universe tab. With CONVERT_TO_BASE the universe prices (and so the optimiser,
# backtest and risk tabs) are converted into it; otherwise they stay in each
# asset's own quote currency.
BASE_CURRENCY = 'USD'
CONVERT_TO_BASE = True
BASE_CURRENCY_CHOICES = ('USD', 'EUR', 'GBP', 'CHF', 'JPY', 'HUF', 'CAD', 'AUD', 'SEK', 'NOK', 'DKK', 'PLN', 'CZK')
# Currency the values in Positions.xlsx are expressed in.
POSITIONS_CURRENCY = 'USD'
# Fiat currencies recognised as the suffix of a Market and as cash lines.
FIAT_CURRENCIES = ('USD', 'EUR', 'GBP', 'CHF', 'JPY', 'CAD', 'AUD', 'HKD', 'SEK', 'NOK',
                   'DKK', 'SGD', 'HUF', 'PLN', 'CZK', 'NZD', 'ZAR', 'ILS', 'CNY', 'INR')

# Asset code (the 'asset' column / the Market without its currency) ->
# yfinance ticker. Anything not listed maps to itself (XLK -> XLK). Add
# entries where the code differs from Yahoo, e.g. {'GOLD': 'GOLD.PA'}.
TICKER_MAP = {}

# Trading currency of tickers that are held but never appear in the trade
# history (otherwise taken from the trades; default BASE_CURRENCY).
CURRENCY_MAP = {}

# Reference asset used for the benchmark line in charts and metrics
BENCHMARK_TICKER = '^GSPC'
BENCHMARK_LABEL = 'Benchmark'

# Cash columns / assets: excluded from weights, still counted in 'Total'.
CASH_COLUMNS = set(FIAT_CURRENCIES) | {'Cash'}


def asset_to_ticker(asset):
    asset = str(asset).strip()
    return TICKER_MAP.get(asset, asset)


def market_to_asset(market):
    """'XLKUSD' -> ('XLK', 'USD'); 'GOLD.PAEUR' -> ('GOLD.PA', 'EUR').

    Returns (market, '') when the Market doesn't end in a known fiat code.
    """
    market = str(market).strip()
    for ccy in FIAT_CURRENCIES:
        if market.endswith(ccy) and len(market) > len(ccy):
            return market[:-len(ccy)], ccy
    return market, ''


def _read_excel_source(path, **kwargs):
    """Read an Excel file from a local path or an http(s) URL."""
    if str(path).lower().startswith(('http://', 'https://')):
        response = requests.get(path, timeout=30)
        response.raise_for_status()          # HTTPError for 4xx / 5xx
        return pd.read_excel(BytesIO(response.content), **kwargs)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return pd.read_excel(path, **kwargs)


def load_excel(path, index_col=0):
    """Read a dated Excel sheet (local path or URL). Returns None on failure."""
    if not path:
        return None
    try:
        df = _read_excel_source(path, index_col=index_col)
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error while downloading {path}: {e}")
        return None
    except RequestException as e:
        print(f"Request failed for {path}: {e}")
        return None
    except FileNotFoundError:
        print(f"File not found: {path}")
        return None
    except ValueError as e:
        print(f"Invalid Excel file at {path}: {e}")
        return None

    df.index = pd.to_datetime(df.index, errors='coerce')
    df = df[df.index.notna()]
    df = df.apply(pd.to_numeric, errors='coerce').sort_index()
    return df[~df.index.duplicated(keep='last')]


_AMOUNT_RE = re.compile(r'^\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)\s*(.*?)\s*$')


def split_amount(value):
    """'12.5000XLK' -> (12.5, 'XLK'); plain numbers -> (x, '')."""
    if isinstance(value, (int, float, np.number)) and not pd.isna(value):
        return float(value), ''
    m = _AMOUNT_RE.match(str(value).replace(',', ''))
    if not m:
        return np.nan, ''
    return float(m.group(1)), m.group(2)


def read_current_portfolio(path):
    """Binance user_asset() layout -> Series of units per ticker.

    Units = free + locked (same as get_inventory). The asset code is mapped to
    the yfinance ticker through TICKER_MAP; cash assets keep their code.
    """
    if not path:
        return pd.Series(dtype=float)
    try:
        df = _read_excel_source(path)
    except Exception as e:
        print(f"❌ Could not read {path}: {e}")
        return pd.Series(dtype=float)
    df.columns = [str(c).strip() for c in df.columns]
    if 'asset' not in df.columns:
        print(f"❌ {path}: no 'asset' column.")
        return pd.Series(dtype=float)
    units = sum(pd.to_numeric(df[c], errors='coerce').fillna(0)
                for c in ('free', 'locked') if c in df.columns)
    assets = df['asset'].astype(str).str.strip()
    tickers_ = [a if a in CASH_COLUMNS else asset_to_ticker(a) for a in assets]
    out = pd.Series(np.asarray(units, dtype=float), index=tickers_)
    out = out.groupby(level=0).sum()
    return out[out.abs() > 1e-12]


def read_trade_history(path):
    """Trade history export -> tidy trades, oldest first.

    Expected columns: Date(UTC), Market, Type, Price, Amount, Total, Fee,
    Fee Coin (the newer layout Pair, Side, Executed "12.5XLK", Amount
    "3.1USD", Fee "0.01USD" is accepted too).

    Output columns: Date, Market, Asset, Ticker, Currency, Side, Price,
    Quantity, Notional, Fee -- money columns in the trade's own Currency.
      * Market 'GOLD.PAEUR' -> Asset 'GOLD.PA' (-> Ticker via TICKER_MAP),
        Currency 'EUR'. A Market without a known fiat suffix is skipped.
      * A fee in the trading currency (or with no Fee Coin) is a cash fee; a
        fee paid in units of the asset reduces the units bought / is valued
        at the trade price on a sale; a fee in another fiat currency is
        converted at the trade's implied rate only if 'Fee FX' is given,
        otherwise it counts as 0 and is flagged.
    """
    if not path:
        return pd.DataFrame()
    try:
        raw = _read_excel_source(path)
    except Exception as e:
        print(f"❌ Could not read {path}: {e}")
        return pd.DataFrame()
    raw.columns = [str(c).strip() for c in raw.columns]
    legacy = 'Market' in raw.columns

    rows, unvalued_fees, skipped = [], 0, set()
    for r in raw.to_dict('records'):
        market = str(r.get('Market') if legacy else r.get('Pair', '')).strip()
        asset, ccy = market_to_asset(market)
        if not ccy:
            skipped.add(market)
            continue
        side = str(r.get('Type') if legacy else r.get('Side', '')).strip().upper()
        price = float(pd.to_numeric(r.get('Price'), errors='coerce'))

        if legacy:
            qty = float(pd.to_numeric(r.get('Amount'), errors='coerce'))
            notional = float(pd.to_numeric(r.get('Total'), errors='coerce'))
            fee = float(pd.to_numeric(r.get('Fee', 0), errors='coerce'))
            fee_coin = str(r.get('Fee Coin', '') or '').strip()
        else:
            qty, unit = split_amount(r.get('Executed'))
            notional, _ = split_amount(r.get('Amount'))
            fee, fee_coin = split_amount(r.get('Fee', 0))
            asset = unit or asset
        if pd.isna(notional):
            notional = qty * price
        fee = 0.0 if pd.isna(fee) else fee
        if fee_coin.lower() == 'nan':
            fee_coin = ''

        fee_cash = 0.0
        if fee_coin in ('', ccy):
            fee_cash = fee
        elif fee_coin == asset:
            if side == 'BUY':
                qty -= fee                          # fee taken out of the units bought
            else:
                fee_cash = fee * price
        elif fee:
            rate = pd.to_numeric(r.get('Fee FX'), errors='coerce')
            if pd.isna(rate):
                unvalued_fees += 1
            else:
                fee_cash = fee * float(rate)

        rows.append({'Date': pd.to_datetime(r.get('Date(UTC)'), errors='coerce'),
                     'Market': market, 'Asset': asset, 'Ticker': asset_to_ticker(asset),
                     'Currency': ccy, 'Side': side, 'Price': price, 'Quantity': qty,
                     'Notional': notional, 'Fee': fee_cash})

    if skipped:
        print(f"⚠️ Skipped markets without a known currency suffix {FIAT_CURRENCIES}: "
              f"{', '.join(sorted(skipped))}")
    if unvalued_fees:
        print(f"⚠️ {unvalued_fees} fee(s) paid in another currency without 'Fee FX' were counted as 0.")
    trades = pd.DataFrame(rows)
    if trades.empty:
        return trades
    trades = trades.dropna(subset=['Date', 'Quantity', 'Price'])
    return trades.sort_values('Date', kind='stable').reset_index(drop=True)


def ticker_currencies(trades, tickers_, quote=None):
    """Trade (cost) currency per ticker: CURRENCY_MAP, then the trades, then the
    quote currency (minor units -> major, GBp -> GBP), then BASE_CURRENCY."""
    from_trades = {} if trades is None or trades.empty else trades.groupby('Ticker')['Currency'].last().to_dict()
    quote = quote if quote is not None else pd.Series(dtype=object)
    out = {}
    for t in tickers_:
        if t in CASH_COLUMNS:
            out[t] = t
            continue
        q = quote.get(t)
        q = {'GBp': 'GBP', 'GBX': 'GBP', 'ZAc': 'ZAR', 'ILA': 'ILS'}.get(q, q)
        out[t] = CURRENCY_MAP.get(t) or from_trades.get(t) or q or BASE_CURRENCY
    return pd.Series(out, dtype=object)


def quote_currencies(tickers_, trades=None):
    """Currency yfinance quotes each ticker in (may be a minor unit such as
    'GBp'); falls back to the trade currency, then BASE_CURRENCY. Cash lines
    are their own currency."""
    tickers_ = list(tickers_)
    assets = [t for t in tickers_ if t not in CASH_COLUMNS]
    yahoo = get_currencies(assets) if assets else pd.Series(dtype=object)
    trade_ccy = ticker_currencies(trades, tickers_)
    return pd.Series({t: t if t in CASH_COLUMNS else (yahoo.get(t) or trade_ccy.get(t) or BASE_CURRENCY)
                      for t in tickers_}, dtype=object)


# Your PnL.get_book_cost uses the trade Total without fees. Set True to
# capitalise buy commissions into the cost price instead; either way every fee
# not in cost is reported separately so total P&L still reconciles.
INCLUDE_FEES_IN_BOOK_COST = False


def get_book_cost(trades):
    """Weighted-average book cost per ticker -- port of PnL.get_book_cost.

        new_cost_price = (old_cost_price * old_balance + purchase_cost) / new_balance

    * Trades with the same timestamp are netted first (groupby Date, as in
      the original).
    * Acquisition (net amount > 0): cost price re-weighted.
    * Disposal: balance falls, cost price unchanged; when the balance reaches
      0 the position is closed and the cost basis resets to 0.
    * Realized P&L (see get_pnl) is taken on the same netted events, so book
      cost + realized + unrealized always add up to cash in minus cash out.

    Returns (book_cost, events): book_cost is indexed by trade timestamp, one
    column per ticker, ffilled then 0 before the first trade (same shape as
    the original); events has one row per ticker and timestamp with the cost
    before/after, the balance and the fees not included in cost.
    """
    if trades.empty:
        return pd.DataFrame(), pd.DataFrame()
    t = trades.copy()
    sign = np.where(t['Side'] == 'SELL', -1.0, 1.0)
    t['Amount'] = sign * t['Quantity']
    capitalised = (t['Side'] == 'BUY') & INCLUDE_FEES_IN_BOOK_COST
    t['Total'] = sign * t['Notional'] + np.where(capitalised, t['Fee'], 0.0)
    t['Fee Charged'] = np.where(capitalised, 0.0, t['Fee'])

    grouped = t.groupby(['Ticker', 'Date'])[['Amount', 'Total', 'Fee Charged']].sum().sort_index()

    series, rows = {}, []
    for ticker, ev in grouped.groupby(level=0):
        balance, cost_price, cost_by_date = 0.0, 0.0, {}
        for (_, date), r in ev.iterrows():
            cost_before = cost_price
            if r['Amount'] > 0:
                new_balance = balance + r['Amount']
                if new_balance != 0:
                    cost_price = (cost_price * balance + r['Total']) / new_balance
                balance = new_balance
            else:
                balance += r['Amount']
                if balance <= 1e-12:
                    balance, cost_price = 0.0, 0.0
            # Realized P&L on the *netted* event, consistent with the cost walk:
            # a net acquisition absorbs any same-timestamp sale into the cost
            # price, so nothing is realized; a net disposal realizes its net
            # proceeds minus the cost *before* the event (so a closing sale
            # isn't valued at the reset cost of 0).
            realized = 0.0 if r['Amount'] > 0 else (-r['Total']) + r['Amount'] * cost_before
            cost_by_date[date] = cost_price
            rows.append({'Date': date, 'Ticker': ticker, 'Cost Before': cost_before,
                         'Cost Price': cost_price, 'Balance': balance,
                         'Realized P&L': realized, 'Fees': r['Fee Charged']})
        series[ticker] = pd.Series(cost_by_date)

    book_cost = pd.DataFrame(series).sort_index().ffill().fillna(0)
    return book_cost, pd.DataFrame(rows)


def get_pnl(events):
    """Realized P&L per ticker -- counterpart of PnL.get_pnl.

    Returns (realized_pnl, profit_and_loss): a one-column frame 'Realized PnL'
    per ticker, and the per-event detail (ticker x timestamp).
    """
    if events.empty:
        return pd.DataFrame(columns=['Realized PnL']), pd.DataFrame()
    detail = events.pivot_table(index='Date', columns='Ticker', values='Realized P&L', aggfunc='sum').fillna(0)
    realized = detail.sum().to_frame('Realized PnL')
    return realized, detail


def build_ledger(trades):
    """Everything the app needs from the trades, as end-of-day frames.

    Returns dict with:
      units_daily      -- units held per ticker at the end of each trade day
      book_cost        -- per trade timestamp (as get_book_cost)
      book_cost_daily  -- cost price at end of each trade day
      realized_daily   -- cumulative realized P&L per ticker (trading currency)
      fees_daily       -- cumulative fees not in cost per ticker
      realized_pnl     -- get_pnl's 'Realized PnL' table
      net_quantities   -- units implied by the trades
    """
    book_cost, events = get_book_cost(trades)
    if events.empty:
        empty = pd.DataFrame()
        return dict(book_cost=empty, book_cost_daily=empty, realized_daily=empty, fees_daily=empty,
                    realized_pnl=pd.DataFrame(columns=['Realized PnL']), net_quantities=pd.Series(dtype=float),
                    units_daily=empty)
    ev = events.assign(Day=events['Date'].dt.normalize())
    cost_daily = ev.pivot_table(index='Day', columns='Ticker', values='Cost Price', aggfunc='last').ffill().fillna(0)
    realized_daily = ev.pivot_table(index='Day', columns='Ticker', values='Realized P&L', aggfunc='sum').fillna(0).cumsum()
    fees_daily = ev.pivot_table(index='Day', columns='Ticker', values='Fees', aggfunc='sum').fillna(0).cumsum()
    realized_pnl, _ = get_pnl(events)
    net_q = ev.sort_values('Date').groupby('Ticker')['Balance'].last()
    units_daily = ev.pivot_table(index='Day', columns='Ticker', values='Balance', aggfunc='last').ffill().fillna(0)
    return dict(book_cost=book_cost, book_cost_daily=cost_daily, realized_daily=realized_daily,
                fees_daily=fees_daily, realized_pnl=realized_pnl, net_quantities=net_q,
                units_daily=units_daily)


def ffill_align(frame, index):
    """Rows of `frame` as of each date in `index` (last known value carried forward).

    Sorts and de-duplicates the source first, so it never hits pandas'
    "index must be monotonic" / "duplicate labels" errors, and never applies
    the fill to the columns.
    """
    f = frame.sort_index()
    f = f[~f.index.duplicated(keep='last')]
    return f.reindex(f.index.union(index)).ffill().reindex(index)


def asof_frame(frame, index, columns, fill=np.nan):
    """Value of `frame` as of each date in `index` (last known row, ffilled)."""
    if frame is None or frame.empty:
        return pd.DataFrame(fill, index=index, columns=columns)
    f = frame.reindex(columns=columns)
    f = f.reindex(f.index.union(index)).sort_index().ffill().reindex(index)
    return f.fillna(fill) if not pd.isna(fill) else f


def pnl_coverage_report(quantities, prices, ledger):
    """Explain P&L lines that can't be computed (they would otherwise show as 0)."""
    held = [c for c in quantities.columns
            if c not in CASH_COLUMNS and (quantities[c].abs() > 1e-12).any()]
    traded = set(ledger['book_cost_daily'].columns) if not ledger['book_cost_daily'].empty else set()
    msgs = []
    no_price = [t for t in held if t not in prices.columns or prices[t].isna().all()]
    no_trades = [t for t in held if t not in traded]
    not_held = sorted(traded - set(quantities.columns))
    if no_price:
        msgs.append(f"⚠️ No price series for {', '.join(no_price)} -- their unrealized P&L is 0.")
    if no_trades:
        msgs.append(f"⚠️ Held in Quantities.xlsx but no trades mapped to {', '.join(no_trades)} "
                    f"-- no book cost, unrealized P&L is 0. Check the Market names / TICKER_MAP.")
    if not_held:
        msgs.append(f"ℹ️ Traded but not a column of Quantities.xlsx: {', '.join(not_held)} "
                    f"(realized P&L on these isn't in the P&L history).")
    if held and len(no_price) + len(no_trades) >= len(held):
        msgs.append("❌ No held asset has both a price and a book cost, so the P&L charts are flat at 0.")
    return msgs


def pnl_history_from_ledger(quantities, prices_base, avg_cost, realized, fx=None, trade_ccy=None, fees=None):
    """Daily total P&L per asset in BASE_CURRENCY.

    unrealized = units x (price in base - book cost x FX of the trade currency)
    plus realized to date and minus fees not in cost, both translated at the
    day's FX. `prices_base` must already be in BASE_CURRENCY (see
    to_base_currency), which also takes care of quotes in pence (GBp) when the
    trades are in GBP. `fx`: date x currency, BASE_CURRENCY per 1 unit.
    """
    tick = [c for c in quantities.columns if c not in CASH_COLUMNS and c in prices_base.columns]
    idx = quantities.index
    px_ = asof_frame(prices_base, idx, tick)
    ac = asof_frame(avg_cost, idx, tick)
    rl = asof_frame(realized, idx, tick, fill=0.0)
    fe = asof_frame(fees, idx, tick, fill=0.0) if fees is not None and not fees.empty else 0.0
    rate = pd.DataFrame(1.0, index=idx, columns=tick)
    if fx is not None and trade_ccy is not None:
        rates = asof_frame(fx, idx, list(fx.columns)).bfill()
        for t in tick:
            ccy = trade_ccy.get(t, BASE_CURRENCY)
            if ccy in rates.columns:
                rate[t] = rates[ccy].fillna(1.0)
    unrealized = (quantities[tick] * (px_ - ac * rate)).where(quantities[tick].abs() > 1e-12, 0.0).fillna(0.0)
    pnl = unrealized + (rl - fe) * rate
    pnl['Total'] = pnl.sum(axis=1)
    return pnl


def return_contributions(weights, returns):
    """Cumulative contribution of each asset to a portfolio's compounded return.

    weights: holdings earning each day's return (i.e. already lagged one day);
    returns: simple daily returns, same index/columns. Each day's contribution
    w_i,t x r_i,t is scaled by the portfolio's growth up to the day before, so

        sum_i contribution_i(T) = V_T / V_0 - 1     (exactly)

    -- the per-asset lines add up to the cumulative return, and fund minus
    benchmark adds up to the gap between their cumulative returns.
    """
    daily = (weights * returns).fillna(0.0)
    growth = (1 + daily.sum(axis=1)).cumprod().shift(1).fillna(1.0)
    contrib = daily.mul(growth, axis=0).cumsum()
    contrib['Total Return'] = contrib.sum(axis=1)
    return contrib


def compute_per_asset_betas(returns_window, benchmark_return_series):
    """Beta of each asset in `returns_window` vs. a benchmark *return* series.

    Identical to the OLS slope with intercept, computed as cov / var.
    Returns a numpy array in `returns_window` column order.
    """
    X = returns_window.fillna(0)
    if X.shape[0] < 2 or X.shape[1] == 0:
        return np.zeros(X.shape[1])
    y = pd.Series(benchmark_return_series).reindex(X.index).fillna(0)
    yc = y - y.mean()
    var_y = (yc ** 2).sum()
    if var_y == 0:
        return np.zeros(X.shape[1])
    Xc = X - X.mean()
    return (Xc.mul(yc, axis=0).sum() / var_y).to_numpy()


def compute_rolling_per_asset_betas(returns_window, benchmark_return_series, window):
    """Rolling per-asset beta vs. a benchmark return series (rolling cov / var)."""
    common_index = returns_window.index.intersection(benchmark_return_series.index)
    returns_aligned = returns_window.loc[common_index]
    benchmark_aligned = benchmark_return_series.loc[common_index]
    rolling_cov = returns_aligned.rolling(window).cov(benchmark_aligned)
    rolling_var = benchmark_aligned.rolling(window).var()
    return rolling_cov.div(rolling_var, axis=0)


def set_choices(dropdown, options, preferred=None):
    """Set a Dropdown's options and value without ever raising
    'TraitError: Invalid selection: value not found'.

    Options are de-duplicated and made plain strings (column labels can be
    tuples or numpy strings). The user's current choice is kept whenever it
    is still an option; otherwise the value is the first of `preferred` (a
    value or list of values) that exists, else the first option, else None.
    """
    opts = list(dict.fromkeys(str(o) if not isinstance(o, str) else o for o in options))
    prefs = preferred if isinstance(preferred, (list, tuple)) else [preferred]
    current = dropdown.value
    if list(dropdown.options) != opts:
        dropdown.options = opts
    for candidate in [current, *prefs]:
        if candidate is not None and candidate in opts:
            dropdown.value = candidate
            return
    dropdown.value = opts[0] if opts else None


def dark_layout(fig, pct_axis=False, **kwargs):
    fig.update_layout(plot_bgcolor="black", paper_bgcolor="black", font_color="white", **kwargs)
    if pct_axis:
        fig.update_layout(yaxis_tickformat=".2%")
    fig.update_traces(textfont=dict(family="Arial Narrow", size=15))
    return fig


def only_show(fig, names):
    """Traces not in `names` start hidden (click the legend to show them)."""
    names = set(names)
    fig.update_traces(visible="legendonly", selector=lambda t: t.name not in names)
    return fig


def display_crypto_app(positions_path=DEFAULT_POSITIONS_PATH,
                       quantities_path=DEFAULT_QUANTITIES_PATH,
                       current_portfolio_path=DEFAULT_CURRENT_PORTFOLIO_PATH,
                       trades_path=DEFAULT_TRADES_PATH,
                       git=None, git_folder=''):
    """git / git_folder: your GitHub client and the repo folder of the files.
    When the Quantities / Positions paths are links, 'Update Files' pushes them
    with git.push_or_update_file(df, <file name>, folder=git_folder)."""
    # =========================================================================
    # Constants / shared state
    # =========================================================================
    dico_strategies = {
        'Minimum Variance': 'minimum_variance',
        'Risk Parity': 'risk_parity',
        'Sharpe Ratio': 'sharpe_ratio',
        'Maximum Diversification': 'maximum_diversification',
        'Eigen Strategy': 'eigenportfolio'}
    options_strat = list(dico_strategies.keys())
    RISK_TYPES_IMPLEMENTED = {'Beta', 'Volatility'}

    global tickers_dataframe, tickers, dataframe, returns_to_use, prices
    global rolling_optimization, performance_pct, performance_fund, dates_end
    global quantities, quantities_core, quantities_overlay, cumulative_results, global_returns
    global holding_tickers, current_weights, fund_names, grid, results
    global var_scenarios, cvar_scenarios, fund_results, grid_stress, grid_mean_shock
    global result_var, result_cvar, quantities_eigen, perf_index_eigen
    global daily_pnl, pnl_history, historical_ptf, performance_ex_post, positions, quantities_holding
    global current_holdings, trades, book_cost, realized_pnl, trade_net_quantities, ledger
    global prices_local, universe_currencies, positions_raw
    global results_vol, series_dict, current_underlying_returns, results_tracking_error, spread_weights

    universe = {
        'Commodities': ['ISLN.L', 'GOLD.PA', 'BZ=F'],
        'Bonds': ['IEF', 'IEI'],
        'Credit': ['DHY', 'LQD'],
        'Equity': ['^GSPC', 'IDU', 'XLK', 'SOXX', 'XLP', 'XLY', 'IYJ', 'MXI', 'IYE',
                   'PICK', 'EXV5.DE', 'IUHC.L', 'IYG', 'XSW'],
    }
    tickers_dataframe = pd.DataFrame(
        [(t, asset_class) for asset_class, lst in universe.items() for t in lst],
        columns=['Ticker', 'Asset Class']).set_index('Ticker')
    tickers_dataframe = tickers_dataframe[~tickers_dataframe.index.duplicated()]

    tickers = []
    holding_tickers = []
    fund_names = []
    dates_end = []
    results = {}
    dataframe = pd.DataFrame()
    prices = pd.DataFrame()
    returns_to_use = pd.DataFrame()
    cumulative_results = pd.DataFrame()
    global_returns = pd.DataFrame()
    current_weights = pd.DataFrame()
    rolling_optimization = pd.DataFrame()
    quantities = pd.DataFrame()
    quantities_core = pd.DataFrame()
    quantities_overlay = pd.DataFrame()
    performance_pct = pd.DataFrame()
    performance_fund = pd.DataFrame()

    daily_pnl = pd.DataFrame()
    pnl_history = pd.DataFrame()
    historical_ptf = pd.DataFrame()
    performance_ex_post = pd.DataFrame()
    positions = pd.DataFrame()
    quantities_holding = pd.DataFrame()
    current_holdings = pd.Series(dtype=float)
    trades = pd.DataFrame()
    book_cost = pd.DataFrame()
    realized_pnl = pd.DataFrame()
    trade_net_quantities = pd.Series(dtype=float)
    ledger = build_ledger(pd.DataFrame())
    prices_local = pd.DataFrame()          # universe prices in each quote currency
    universe_currencies = pd.Series(dtype=object)
    positions_raw = pd.DataFrame()         # Positions.xlsx as read, in POSITIONS_CURRENCY

    constraint_container = {'constraints': [], 'allocation_df': pd.DataFrame()}
    price_cache = {}

    # =========================================================================
    # Constraint builders
    # =========================================================================
    def risk_constraints_frame(rows):
        df = pd.DataFrame(rows)
        if df.empty or 'Risk' not in df.columns:
            return pd.DataFrame(columns=['Risk', 'Sign', 'Limit'])
        return df.drop_duplicates(subset=['Risk'], keep='last')

    def build_beta_risk_constraints(returns_window, benchmark_level_series, risk_constraint_dataframe):
        """Beta constraints for risk.optimize(); [] if there is no 'Beta' row."""
        if 'Risk' not in risk_constraint_dataframe.columns:
            return []
        beta_rows = risk_constraint_dataframe[risk_constraint_dataframe['Risk'] == 'Beta']
        if beta_rows.empty:
            return []
        beta = compute_per_asset_betas(returns_window, benchmark_level_series.pct_change(fill_method=None))
        beta_data = beta_rows.iloc[0]
        return beta_constraint(beta, beta_data['Sign'], beta_data['Limit'])

    def build_vol_risk_constraints(returns_window, risk_constraint_dataframe):
        """Volatility constraints for risk.optimize(); [] if there is no 'Volatility' row."""
        if 'Risk' not in risk_constraint_dataframe.columns:
            return []
        vol_rows = risk_constraint_dataframe[risk_constraint_dataframe['Risk'] == 'Volatility']
        if vol_rows.empty:
            return []
        vol_data = vol_rows.iloc[0]
        return vol_constraint(returns_window.cov(), vol_data['Sign'], vol_data['Limit'])

    # =========================================================================
    # INVESTMENT UNIVERSE TAB
    # =========================================================================
    start_date = widgets.DatePicker(
        value=datetime.date(2020, 1, 1), description='Starting Date of Backtest',
        style={'description_width': '200px'}, layout=widgets.Layout(width='350px'))

    loading_bar = widgets.IntProgress(description='Loading prices...', min=0, max=100, style={'description_width': '150px'})
    data_button = widgets.Button(description='Get Prices', button_style='info')
    base_currency_dd = widgets.Dropdown(
        description='Base Currency', options=list(dict.fromkeys((BASE_CURRENCY,) + tuple(BASE_CURRENCY_CHOICES))),
        value=BASE_CURRENCY, style={'description_width': '120px'}, layout=widgets.Layout(width='230px'))
    convert_checkbox = widgets.Checkbox(value=CONVERT_TO_BASE, description='Convert prices to base currency',
                                        style={'description_width': 'initial'})
    scope_output = widgets.Output()
    strategy_output = widgets.Output()
    main_output = widgets.Output()
    output_returns = widgets.Output()
    constraint_output = widgets.Output()
    price_output = widgets.Output()

    dropdown_asset1 = widgets.Dropdown(description='Asset 1:', style={'description_width': '150px'})
    dropdown_asset2 = widgets.Dropdown(description='Asset 2:', style={'description_width': '150px'})

    unchecked = set()          # tickers the user untick -- kept across refreshes
    details_state = {'show': False}

    def scope_update(_=None):
        """Redraw the universe table and its checkboxes (tick = include in 'Get Prices')."""
        selected = tickers_dataframe
        selected_list = list(selected.index)
        table = selected
        if details_state['show']:
            table = selected.join(fetch_ticker_details(selected_list))
        with scope_output:
            scope_output.clear_output(wait=True)
            display(display_scrollable_df(table))
            checkboxes = {t: widgets.Checkbox(description=t, value=t not in unchecked) for t in selected_list}
            rows = [widgets.HBox(list(checkboxes.values())[i:i + 5]) for i in range(0, len(checkboxes), 5)]

            def on_change(change=None):
                global tickers
                for k, cb in checkboxes.items():
                    (unchecked.discard if cb.value else unchecked.add)(k)
                tickers = [k for k, cb in checkboxes.items() if cb.value]

            for cb in checkboxes.values():
                cb.observe(on_change, names="value")
            on_change()
            display(Markdown("### Selected Tickers"))
            display(widgets.VBox(rows))

    scope_update()

    # --- search Yahoo Finance and add tickers to the universe ---
    search_box = widgets.Text(continuous_update=False,   # value only changes on Enter / leaving the box
                              placeholder='Name or ticker, e.g. "Nvidia", "gold ETF", "EXV5.DE"',
                              description='Search', layout=widgets.Layout(width='450px'))
    search_button = widgets.Button(description='Search', button_style='info', icon='search')
    search_results = widgets.Dropdown(description='Result', options=[], layout=widgets.Layout(width='650px'))
    asset_class_choice = widgets.Combobox(
        description='Asset Class', placeholder='Auto (from Yahoo type)',
        options=sorted(set(tickers_dataframe['Asset Class']) | set(QUOTE_TYPE_CLASS.values())),
        ensure_option=False)
    add_ticker_button = widgets.Button(description='Add to Universe', button_style='success', icon='plus')
    details_button = widgets.Button(description='Load Details', icon='info-circle',
                                    tooltip='Name, exchange, currency, volume, market cap... from yfinance')
    search_output = widgets.Output()
    search_hits = {}

    def on_search(_=None):
        search_hits.clear()
        with search_output:
            search_output.clear_output()
            hits = search_yahoo(search_box.value)
            if hits.empty:
                search_results.options = []
                print(f"No match for '{search_box.value}'.")
                return
            for r in hits.itertuples(index=False):
                label = f"{r.Symbol} — {r.Name} ({r.Type}, {r.Exchange})"
                search_hits[label] = r._asdict()
            set_choices(search_results, list(search_hits))

    def on_add_ticker(_=None):
        global tickers_dataframe
        with search_output:
            search_output.clear_output()
            hit = search_hits.get(search_results.value)
            symbol = hit['Symbol'] if hit else search_box.value.strip().upper()
            if not symbol:
                print("Search for a ticker first.")
                return
            if symbol in tickers_dataframe.index:
                print(f"{symbol} is already in the universe.")
                return
            # only add what yfinance can actually price
            if not has_price_history(symbol):
                print(f"❌ yfinance returns no prices for {symbol} -- not added.")
                return
            quote_type = (hit or {}).get('Type') or fetch_ticker_details([symbol]).iloc[0].get('Type') or ''
            asset_class = asset_class_choice.value.strip() or QUOTE_TYPE_CLASS.get(str(quote_type).upper(), 'Other')
            tickers_dataframe = pd.concat(
                [tickers_dataframe, pd.DataFrame({'Asset Class': [asset_class]}, index=pd.Index([symbol], name='Ticker'))])
            asset_class_choice.options = sorted(set(asset_class_choice.options) | {asset_class})
            print(f"✅ Added {symbol} ({asset_class}). Click 'Get Prices' to load it.")
        scope_update()

    def on_details(_=None):
        details_state['show'] = True
        with search_output:
            search_output.clear_output()
            print("Loading details from yfinance...")
        scope_update()
        with search_output:
            search_output.clear_output()

    search_box.observe(lambda ch: on_search() if ch['new'].strip() else None, names='value')
    search_button.on_click(on_search)
    add_ticker_button.on_click(on_add_ticker)
    details_button.on_click(on_details)
    search_ui = widgets.VBox([
        widgets.HBox([search_box, search_button]),
        widgets.HBox([search_results]),
        widgets.HBox([asset_class_choice, add_ticker_button, details_button]),
        search_output])

    def get_price_threading(tickers_list, start):
        """Daily adjusted closes (Yahoo_API.get_close) in each ticker's quote
        currency, with a progress bar and a per-session cache."""
        tickers_list = sorted({t for t in tickers_list if t and t not in CASH_COLUMNS and t != 'Total'})
        if not tickers_list:
            return pd.DataFrame()
        if isinstance(start, datetime.datetime):
            start = start.date()
        today = datetime.date.today()
        if start >= today:
            print("Start date must be in the past.")
            return pd.DataFrame()

        key = (tuple(tickers_list), start, today)
        if key in price_cache:
            return price_cache[key].copy()

        loading_bar.value = 0
        loading_bar.max = len(tickers_list)
        display(loading_bar)

        def progress(_ticker, _ok):
            loading_bar.value += 1

        price = get_close(tickers_list, start=start, end=today, max_workers=8, on_progress=progress)
        if price.attrs.get('failed'):
            print(f"⚠️ No data returned for: {', '.join(price.attrs['failed'])}")
        if price.empty:
            return pd.DataFrame()
        price_cache[key] = price
        return price.copy()

    def prices_to_base(local_prices, start):
        """Local-currency prices -> BASE_CURRENCY (no-op when CONVERT_TO_BASE is off)."""
        ccys = quote_currencies(local_prices.columns, trades)
        if not CONVERT_TO_BASE or local_prices.empty:
            return local_prices, ccys
        fx = get_fx_rates(ccys.values, BASE_CURRENCY, start=start)
        if fx.attrs.get('missing'):
            print(f"⚠️ No FX rate for {', '.join(fx.attrs['missing'])} -> {BASE_CURRENCY}; left unconverted")
        return to_base_currency(local_prices, ccys, fx), ccys

    def render_universe_charts():
        range_slice = slice(start_date_perf.value, end_date_perf.value)
        with price_output:
            price_output.clear_output(wait=True)
            price_graph = widgets.Output()
            return_graph = widgets.Output()
            with price_graph:
                fig = px.line(dataframe.loc[range_slice], title="Price", width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig), [BENCHMARK_TICKER]).show()
            with return_graph:
                cumulative_returns = returns_to_use.loc[range_slice].copy()
                if not cumulative_returns.empty:
                    cumulative_returns.iloc[0] = 0
                cumulative_returns = (1 + cumulative_returns).cumprod() * 100
                fig2 = px.line(cumulative_returns, title="Cumulative Performance", width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig2), [BENCHMARK_TICKER]).show()
            display(widgets.HBox([price_graph, return_graph]))
            display(display_scrollable_df(dataframe))

    def get_prices(_=None):
        global prices, dataframe, returns_to_use, prices_local, universe_currencies

        with main_output:
            main_output.clear_output(wait=True)
            if not tickers:
                print("No tickers selected.")
                return
            start = start_date.value
            if not isinstance(start, datetime.date):
                print("Please select a valid start date.")
                return

            price_cache.clear()
            scope_prices = get_price_threading(tickers, start)
            if scope_prices.empty:
                print("❌ No prices could be loaded.")
                return

            prices_local = scope_prices
            prices, universe_currencies = prices_to_base(scope_prices, start)
            if CONVERT_TO_BASE:
                foreign = universe_currencies[universe_currencies != BASE_CURRENCY]
                if len(foreign):
                    print(f"💱 Converted to {BASE_CURRENCY}: " +
                          ", ".join(f"{t} ({c})" for t, c in foreign.items()))
            else:
                mixed = sorted(set(universe_currencies.dropna()))
                if len(mixed) > 1:
                    print(f"⚠️ Prices left in local currencies ({', '.join(mixed)}) -- "
                          "returns and risk mix currencies.")
            # simple returns: (1+r).cumprod() and weighted sums across assets are only exact with these
            returns = prices.pct_change(fill_method=None)

            valid_cols = returns.columns[returns.isna().sum() < 30]
            if len(valid_cols) == 0:
                print("❌ Every asset has too much missing history for this start date.")
                return
            returns_to_use = returns[valid_cols].sort_index()
            returns_to_use = returns_to_use[~returns_to_use.index.duplicated(keep="first")]

            dataframe = prices[valid_cols].sort_index().dropna()
            main_output.clear_output()

            dropped = sorted(set(prices.columns) - set(valid_cols))
            if dropped:
                print(f"⚠️ Dropped (too much missing history): {', '.join(dropped)}")

            cols = list(dataframe.columns)
            set_choices(dropdown_asset, cols + ["All"])
            set_choices(dropdown_asset1, cols, cols[0])
            set_choices(dropdown_asset2, cols, cols[1:2] or cols[:1])

            print(f"✅ Loaded prices for {len(cols)} assets "
                  f"from {dataframe.index[0].date()} to {dataframe.index[-1].date()}")
            display(display_scrollable_df(get_asset_returns(dataframe)))
            display(display_scrollable_df(get_asset_risk(dataframe)))

        render_universe_charts()
        # Old constraints may reference tickers no longer in scope.
        on_clear_constraints(None)
        on_clear_overlays(None)
        on_clear_risk(None)
        refresh_trajectory_choices()

    data_button.on_click(get_prices)
    start_date.observe(lambda ch: get_prices() if ch['new'] else None, names='value')

    # =========================================================================
    # STRATEGY TAB -- constraints, overlays, risk limits, allocation grid
    # =========================================================================
    dropdown_asset = widgets.Dropdown(description='Asset:', options=['All'], value=None)
    dropdown_sign = widgets.Dropdown(description='Sign:', options=["=", "≥", "≤"])
    dropdown_limit = widgets.FloatText(description='Limit')
    add_constraint_btn = widgets.Button(description='Add Constraint', button_style='success')
    clear_constraints_btn = widgets.Button(description='Clear All', button_style='danger')
    constraints = []

    selected_fund = widgets.Dropdown(description="Fund:")
    selected_bench = widgets.Dropdown(description="Bench:")
    selected_fund_var = widgets.Dropdown(description="Fund:")
    options_risk = ['Beta', 'Volatility']

    add_strategy_btn = widgets.Button(description='Add Strategy')
    clear_strategy_btn = widgets.Button(description='Clear Strategy')
    add_risk_btn = widgets.Button(description='Add Risk Strat')
    clear_risk_btn = widgets.Button(description='Clear Risk Strat')

    dropdown_strategy_limit = widgets.FloatText(description='Overlay Limit', style={'description_width': '150px'})
    strat_overlay = widgets.Dropdown(description='Strategy Overlay', options=options_strat, value='Minimum Variance', style={'description_width': '150px'})
    risk_type = widgets.Dropdown(description='Risk Type', value=None, options=options_risk, style={'description_width': '150px'})
    risk_limit = widgets.FloatText(description='Risk Limit', value=0.0, style={'description_width': '150px'})
    dropdown_risk_sign = widgets.Dropdown(description='Sign:', options=["=", "≥", "≤"], style={'description_width': '150px'})
    overlays = []
    overlay_output = widgets.Output()

    def on_add_overlays(_):
        overlays.append({'Strategy': strat_overlay.value, 'Limit': dropdown_strategy_limit.value})
        with overlay_output:
            overlay_output.clear_output(wait=True)
            display(display_scrollable_df(pd.DataFrame(overlays)))

    def on_clear_overlays(_):
        overlays.clear()
        with overlay_output:
            overlay_output.clear_output(wait=True)
            display(display_scrollable_df(pd.DataFrame(columns=['Strategy', 'Limit'])))

    add_strategy_btn.on_click(on_add_overlays)
    clear_strategy_btn.on_click(on_clear_overlays)

    risk_constraints = []
    risk_constraints_output = widgets.Output()

    def on_add_risk(_):
        with risk_constraints_output:
            risk_constraints_output.clear_output(wait=True)
            if risk_type.value is None:
                print("⚠️ Select a Risk Type before adding.")
            elif risk_type.value not in RISK_TYPES_IMPLEMENTED:
                print(f"⚠️ '{risk_type.value}' isn't implemented in the optimizer.")
            else:
                risk_constraints.append({'Risk': risk_type.value, 'Sign': dropdown_risk_sign.value,
                                         'Limit': risk_limit.value})
            display(display_scrollable_df(risk_constraints_frame(risk_constraints)))

    def on_clear_risk(_):
        risk_constraints.clear()
        with risk_constraints_output:
            risk_constraints_output.clear_output(wait=True)
            display(display_scrollable_df(pd.DataFrame(columns=['Risk', 'Sign', 'Limit'])))

    add_risk_btn.on_click(on_add_risk)
    clear_risk_btn.on_click(on_clear_risk)

    def on_add_constraint_clicked(_):
        with constraint_output:
            constraint_output.clear_output(wait=True)
            if dropdown_asset.value is None:
                print("⚠️ Select an Asset before adding a constraint.")
            else:
                constraints.append({'Asset': dropdown_asset.value,
                                    'Sign': dropdown_sign.value,
                                    'Limit': dropdown_limit.value})
            display(display_scrollable_df(pd.DataFrame(constraints) if constraints
                                          else pd.DataFrame(columns=['Asset', 'Sign', 'Limit'])))

    def on_clear_constraints(_):
        constraints.clear()
        with constraint_output:
            constraint_output.clear_output(wait=True)
            display(display_scrollable_df(pd.DataFrame(columns=['Asset', 'Sign', 'Limit'])))

    add_constraint_btn.on_click(on_add_constraint_clicked)
    clear_constraints_btn.on_click(on_clear_constraints)

    def refresh_allocation_dropdowns():
        """Allocation names in the fund / benchmark dropdowns. Current choices
        are kept; the first time, the benchmark defaults to a different row
        than the fund (Current Weights if there is one)."""
        names = list(grid.data.index)
        if not names:
            return
        other = ['Current Weights', *[n for n in names if n != names[0]]]
        set_choices(selected_fund, names, names[0])
        set_choices(selected_fund_var, names, names[0])
        set_choices(selected_bench, names, other)
        set_choices(benchmark_tracking_error, names, other)
        refresh_trajectory_choices()

    def on_add_click(_):
        if grid.data is None or grid.data.empty:
            return
        label = f"Allocation {grid.data.shape[0]}"
        new_df = pd.DataFrame([np.zeros(grid.data.shape[1])], columns=grid.data.columns, index=[label])
        grid.data = pd.concat([pd.DataFrame(grid.data), new_df])
        refresh_allocation_dropdowns()

    def clear_allocation(_):
        if constraint_container.get('allocation_df') is not None and not constraint_container['allocation_df'].empty:
            grid.data = constraint_container['allocation_df']
            refresh_allocation_dropdowns()

    button_add = widgets.Button(description="Add Allocation")
    button_clear = widgets.Button(description="Clear Allocation")
    button_add.on_click(on_add_click)
    button_clear.on_click(clear_allocation)

    # --- performance date pickers (start = 2 months after backtest start) ---
    if isinstance(start_date.value, datetime.date):
        start_perf_date = (pd.Timestamp(start_date.value) + pd.DateOffset(months=2)).replace(day=1).date()
    else:
        start_perf_date = datetime.date.today() - datetime.timedelta(days=365)
    start_date_perf = widgets.DatePicker(value=start_perf_date, layout=widgets.Layout(width='350px'))
    end_date_perf = widgets.DatePicker(value=datetime.date.today(), layout=widgets.Layout(width='350px'))

    frequency_graph = widgets.Dropdown(description='Frequency:', options=['Year', 'Month'], value='Year')
    benchmark = widgets.Dropdown(description='Benchmark:', options=['Fund', BENCHMARK_LABEL], value=BENCHMARK_LABEL)
    fund = widgets.Dropdown(description='Fund:', options=['Fund', BENCHMARK_LABEL], value='Fund')
    benchmark_tracking_error = widgets.Dropdown(description='Benchmark:')
    perf_output = widgets.Output()
    vol_output = widgets.Output()
    drawdown_output = widgets.Output()
    frontier_output = widgets.Output()

    def parse_range(start_widget, end_widget):
        """(start_ts, end_ts) or None if missing/invalid/reversed."""
        try:
            s = pd.to_datetime(start_widget.value)
            e = pd.to_datetime(end_widget.value)
        except Exception:
            return None
        if pd.isna(s) or pd.isna(e) or s > e:
            return None
        return s, e

    def updated_cumulative_perf(_):
        global performance_pct, cumulative_results, global_returns

        rng = parse_range(start_date_perf, end_date_perf)
        if rng is None:
            for out in (main_output, output_returns):
                with out:
                    out.clear_output()
                    print("⚠️ Invalid date range.")
            for out in (vol_output, perf_output, drawdown_output, frontier_output):
                out.clear_output()
            return
        start_ts, end_ts = rng

        if dataframe.empty:
            with main_output:
                main_output.clear_output(wait=True)
                print("⚠️ Load Prices.")
            return

        range_prices = dataframe.loc[start_ts:end_ts]
        range_returns = range_prices.pct_change(fill_method=None)
        with main_output:
            main_output.clear_output(wait=True)
            display(display_scrollable_df(get_asset_returns(range_prices)))
            display(display_scrollable_df(get_asset_risk(range_prices)))
        render_universe_charts()

        if performance_pct is None or performance_pct.empty:
            with output_returns:
                output_returns.clear_output(wait=True)
                print("⚠️ No performance data available yet. Please run an optimization first.")
            return

        cons = None
        if constraints:
            try:
                cons = build_constraint(dataframe, pd.DataFrame(constraints).to_numpy())
            except Exception as e:
                with output_returns:
                    print("Error building constraints:", e)

        performance_pct.index = pd.to_datetime(performance_pct.index)
        cumulative_performance = performance_pct.loc[start_ts:end_ts].copy()
        if cumulative_performance.empty:
            with output_returns:
                output_returns.clear_output(wait=True)
                print(f"⚠️ No data found for this date range. Available range: "
                      f"{performance_pct.index.min().date()} → {performance_pct.index.max().date()}")
            return
        cumulative_performance.iloc[0] = 0
        cumulative_results = (1 + cumulative_performance).cumprod() * 100

        # allocations started on the backtest's first day, shown over the chart range (rebased to 100)
        portfolio_returns = rebalanced_time_series(dataframe, grid.data, frequency=rebalancing_frequency.value)
        portfolio_returns = portfolio_returns.loc[start_ts:end_ts]
        portfolio_returns = portfolio_returns.div(portfolio_returns.iloc[0]) * 100
        cumulative_results = pd.concat([cumulative_results, portfolio_returns], axis=1)
        global_returns = cumulative_results.pct_change(fill_method=None)

        drawdown = (cumulative_results - cumulative_results.cummax()) / cumulative_results.cummax()
        rolling_vol_ptf = global_returns.rolling(window_vol.value).std() * np.sqrt(252)
        frontier_indicators, fig4 = get_frontier(range_returns, grid.data, cons)
        update_dropdown_options()

        defaults = ["Fund", BENCHMARK_LABEL]
        with output_returns:
            output_returns.clear_output(wait=True)
            display(display_scrollable_df(rebalanced_metrics(cumulative_results)))
            display(display_scrollable_df(get_portfolio_risk(grid.data, range_prices, cumulative_results, benchmark_tracking_error.value)))
            display(display_scrollable_df(frontier_indicators))
        with perf_output:
            perf_output.clear_output(wait=True)
            fig = px.line(cumulative_results, title='Performance', width=800, height=400, render_mode='svg')
            only_show(dark_layout(fig), defaults).show()
        with drawdown_output:
            drawdown_output.clear_output(wait=True)
            fig2 = px.line(drawdown, title='Drawdown', width=800, height=400, render_mode='svg')
            only_show(dark_layout(fig2, pct_axis=True), defaults).show()
        with vol_output:
            vol_output.clear_output(wait=True)
            fig3 = px.line(rolling_vol_ptf, title="Portfolio Rolling Volatility", width=800, height=400, render_mode='svg')
            only_show(dark_layout(fig3, pct_axis=True), defaults).show()
        with frontier_output:
            frontier_output.clear_output(wait=True)
            fig4.update_layout(width=800, height=400, title={'text': "Efficient Frontier"},
                               yaxis_tickformat=".2%", xaxis_tickformat=".2%")
            fig4.update_traces(textfont=dict(family="Arial Narrow", size=15))
            fig4.show()

    refresh_perf_button = widgets.Button(description='Refresh')
    refresh_perf_button.on_click(updated_cumulative_perf)

    # =========================================================================
    # OPTIMIZATION
    # =========================================================================
    rebalancing_frequency = widgets.Dropdown(description='Frequency', options=['Monthly', 'Quarterly', 'Yearly'], value='Monthly')
    strat = widgets.Dropdown(description='Strategy', options=options_strat, value='Minimum Variance')
    window_vol = widgets.IntText(value=252, description='Vol Window:')
    calendar_output = widgets.Output()

    def show_graph(_):
        with calendar_output:
            calendar_output.clear_output(wait=True)
            if cumulative_results.empty or cumulative_results.shape[1] < 2:
                print("⚠️ No performance data available yet. Please run an optimization first.")
                return
            if fund.value == benchmark.value:
                print("⚠️ Benchmark and Fund must be different.")
                return
            graphs = get_calendar_graph(cumulative_results, freq=frequency_graph.value,
                                        benchmark=benchmark.value, fund=fund.value)
            left, right = widgets.Output(), widgets.Output()
            keys = list(graphs.keys())
            with left:
                graphs[keys[0]].show()
                graphs[keys[2]].show()
            with right:
                graphs[keys[1]].show()
                graphs[keys[3]].show()
            display(widgets.HBox([left, right]))

    graph_button = widgets.Button(description='Update Perf', button_style='info')
    graph_button.on_click(show_graph)
    optimize_btn = widgets.Button(description='Optimize Portfolio', button_style='primary')
    grid = DataGrid(pd.DataFrame(), editable=True, layout={"height": "250px"})

    def update_dropdown_options():
        global fund_names
        if cumulative_results.empty:
            return
        options = list(cumulative_results.columns)
        set_choices(fund, options, options[0])
        set_choices(benchmark, options, options[1:2] or options[:1])
        fund_names = list(grid.data.index)
        refresh_allocation_dropdowns()

    def on_optimize_clicked(_):
        nonlocal constraint_container
        with constraint_output:
            constraint_output.clear_output(wait=True)
            if dataframe.empty or returns_to_use.empty:
                print("⚠️ Load price data before optimizing.")
                return

            all_constraints = []
            risk_constraint_dataframe = risk_constraints_frame(risk_constraints)
            cons = None
            if constraints:
                try:
                    cons = build_constraint(dataframe, pd.DataFrame(constraints).to_numpy())
                    all_constraints.extend(cons)
                except Exception as e:
                    print("Error building constraints:", e)

            window_returns = returns_to_use.loc[start_date_perf.value:end_date_perf.value]

            if not risk_constraint_dataframe.empty:
                try:
                    all_constraints.extend(build_vol_risk_constraints(window_returns, risk_constraint_dataframe))
                except Exception as e:
                    print("Error building volatility constraints:", e)

                has_beta = (risk_constraint_dataframe['Risk'] == 'Beta').any()
                if has_beta and not grid.data.empty and benchmark_tracking_error.value in grid.data.index:
                    try:
                        # benchmark held since the backtest start, cut to the optimisation window
                        benchmark_level = rebalanced_portfolio(
                            dataframe, grid.data.loc[benchmark_tracking_error.value],
                            frequency=rebalancing_frequency.value).sum(axis=1).loc[start_date_perf.value:end_date_perf.value]
                        all_constraints.extend(build_beta_risk_constraints(window_returns, benchmark_level, risk_constraint_dataframe))
                    except Exception as e:
                        print("Error building beta constraints:", e)
                elif has_beta:
                    print("⚠️ Skipping beta constraint: no benchmark allocation selected yet.")

            portfolio = RiskAnalysis(window_returns)
            objectives = {
                'sharpe_ratio': 'Optimal Portfolio',
                'minimum_variance': 'Min Variance',
                'maximum_diversification': 'Max Diversification',
                'risk_parity': 'Risk Parity',
                'eigenportfolio': 'Eigen Portfolio',
            }
            constrained_names = {
                'sharpe_ratio': 'Constrained Optimal Portfolio',
                'minimum_variance': 'Constrained Min Var',
                'maximum_diversification': 'Max Diversification Constrained',
                'risk_parity': 'Constrained RP',
                'eigenportfolio': 'Eigen Portfolio Constrained',
            }
            allocation = {}
            for objective, name in objectives.items():
                unconstrained = portfolio.optimize(objective)
                allocation[name] = list(unconstrained)
                constrained = unconstrained
                if all_constraints:
                    try:
                        constrained = portfolio.optimize(objective, constraints=all_constraints)
                    except Exception as e:
                        print(f"Constrained {name} failed, using unconstrained weights: {e}")
                allocation[constrained_names[objective]] = list(constrained)
            n = returns_to_use.shape[1]
            allocation['Equal Weighted'] = [1 / n] * n

            allocation_df = pd.DataFrame(allocation, index=dataframe.columns).T.round(4)
            if not current_weights.empty:
                allocation_df.loc['Current Weights'] = (
                    current_weights['Current Weights'].reindex(dataframe.columns).fillna(0))

            constraint_container = {'constraints_dataframe': list(constraints), 'constraints': cons,
                                    'allocation_df': allocation_df}
            grid.data = allocation_df
            refresh_allocation_dropdowns()

            display(display_scrollable_df(pd.DataFrame(constraints)))

        reset_stress(None)

    def get_result(_):
        global rolling_optimization, performance_pct, performance_fund, dates_end
        global quantities, quantities_core, quantities_overlay, results

        with strategy_output:
            strategy_output.clear_output(wait=True)
            if dataframe.empty or returns_to_use.empty:
                print("⚠️ Load price data before optimizing.")
                return

            risk_constraint_dataframe = risk_constraints_frame(risk_constraints)

            # Benchmark level series computed once, sliced by every worker.
            benchmark_level_full = None
            has_beta = (not risk_constraint_dataframe.empty
                        and (risk_constraint_dataframe['Risk'] == 'Beta').any())
            if has_beta and not grid.data.empty and benchmark_tracking_error.value in grid.data.index:
                benchmark_level_full = rebalanced_portfolio(
                    dataframe, grid.data.loc[benchmark_tracking_error.value],
                    frequency=rebalancing_frequency.value).sum(axis=1)
            elif has_beta:
                print("⚠️ Skipping beta constraint: no benchmark allocation selected yet.")

            cons = None
            if constraints:
                try:
                    cons = build_constraint(dataframe, pd.DataFrame(constraints).to_numpy())
                except Exception as e:
                    print("Error building constraints:", e)

            freq_map = {'Monthly': pd.offsets.BMonthEnd(),
                        'Quarterly': pd.offsets.BQuarterEnd(),
                        'Yearly': pd.offsets.BYearEnd()}
            offset = freq_map.get(rebalancing_frequency.value, pd.offsets.BMonthEnd())
            candidate_anchors = pd.DatetimeIndex(sorted(set(dataframe.index + offset)))
            if candidate_anchors.empty:
                candidate_anchors = pd.DatetimeIndex([returns_to_use.index[-1]])

            idx = np.array(returns_to_use.index.get_indexer(candidate_anchors, method='nearest'))
            idx = idx[idx >= 0]
            selected_dates = returns_to_use.index[idx].tolist()
            dates_end = sorted(set(selected_dates + [returns_to_use.index[-1]]))

            if len(dates_end) < 2:
                print("⚠️ Not enough anchor dates to perform rolling optimization.")
                return

            strategy_key = dico_strategies[strat.value]
            windows = [(returns_to_use.loc[dates_end[i]:dates_end[i + 1]], dates_end[i], dates_end[i + 1])
                       for i in range(len(dates_end) - 1)]
            strategy_keys = {strategy_key} | {dico_strategies[row['Strategy']] for row in overlays}
            all_tasks = [(subset, s, e, k) for subset, s, e in windows for k in strategy_keys]

            results = {}
            results_lock = threading.Lock()

            def worker(subset, start, end, key):
                if subset.empty or len(subset) < 2:
                    return None
                constraints_to_use = list(cons) if cons else []
                try:
                    constraints_to_use.extend(build_vol_risk_constraints(subset, risk_constraint_dataframe))
                except Exception as e:
                    print(f"Vol constraint failed {start} -> {end}: {e}")
                if benchmark_level_full is not None:
                    try:
                        constraints_to_use.extend(build_beta_risk_constraints(subset, benchmark_level_full, risk_constraint_dataframe))
                    except Exception as e:
                        print(f"Beta constraint failed {start} -> {end}: {e}")
                try:
                    risk = RiskAnalysis(subset)
                    if constraints_to_use:
                        opt = risk.optimize(objective=key, constraints=constraints_to_use)
                    else:
                        opt = risk.optimize(objective=key)
                    return subset.index[-1], np.round(opt, 6), key
                except Exception as e:
                    print(f"Optimization failed {start} -> {end}: {e}")
                    return None

            with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
                futures = [executor.submit(worker, *task) for task in all_tasks]
                for future in as_completed(futures):
                    out = future.result()
                    if out is not None:
                        date_key, weights, key = out
                        with results_lock:
                            results.setdefault(key, {})[date_key] = weights

            if strategy_key not in results:
                print("❌ Every optimization window failed for the core strategy.")
                return

            rolling_optimization = pd.DataFrame(results[strategy_key], index=dataframe.columns).T.sort_index()
            total_overlay = pd.DataFrame(0.0, index=rolling_optimization.index, columns=rolling_optimization.columns)
            core_weight = 1.0
            core_strat = rolling_optimization.copy()

            for row in overlays:
                key = dico_strategies[row['Strategy']]
                if key not in results:
                    print(f"⚠️ Overlay '{row['Strategy']}' failed on every window, skipped.")
                    continue
                overlay_df = pd.DataFrame(results[key], index=dataframe.columns).T.sort_index() * row['Limit']
                total_overlay = total_overlay.add(overlay_df, fill_value=0)
                core_weight -= row['Limit']

            if core_weight < -1e-9:
                print("⚠️ Overlay weights exceed 100%")
                return
            core_weight = max(core_weight, 0.0)

            rolling_optimization = rolling_optimization * core_weight + total_overlay

            first_row = pd.Series(1 / len(dataframe.columns), index=dataframe.columns, name=dates_end[0])
            rolling_optimization = pd.concat([pd.DataFrame([first_row]), rolling_optimization])
            core_strat = pd.concat([pd.DataFrame([first_row]), core_strat])
            denom = (1 - core_weight) if core_weight < 1 else 1
            total_overlay = pd.concat([pd.DataFrame([first_row]), total_overlay / denom])

            core_out, overlay_out, final_out = widgets.Output(), widgets.Output(), widgets.Output()
            with core_out:
                display(Markdown("### Core Strategy"))
                display(display_scrollable_df(core_strat))
            with overlay_out:
                display(Markdown("### Tactical Allocation"))
                display(display_scrollable_df(total_overlay))
            with final_out:
                display(Markdown("### Final Strategy"))
                display(display_scrollable_df(rolling_optimization))
            display(widgets.HBox([final_out, core_out, overlay_out]))

            model = rolling_optimization.iloc[[-1]].copy()
            model.index = ['Model']
            if not grid.data.empty:
                model = model.reindex(columns=grid.data.columns).fillna(0)
                grid.data = pd.concat([grid.data.drop(index='Model', errors='ignore'), model], axis=0)
                refresh_allocation_dropdowns()

            quantities = rebalanced_dynamic_quantities(dataframe, rolling_optimization)
            quantities_core = rebalanced_dynamic_quantities(dataframe, core_strat)
            quantities_overlay = rebalanced_dynamic_quantities(dataframe, total_overlay)
            performance_fund = pd.DataFrame({'Fund': (quantities * dataframe).sum(axis=1),
                                             'Core': (quantities_core * dataframe).sum(axis=1),
                                             'Overlay': (quantities_overlay * dataframe).sum(axis=1)})
            if BENCHMARK_TICKER in dataframe.columns:
                performance_fund[BENCHMARK_LABEL] = dataframe[BENCHMARK_TICKER]
            performance_pct = performance_fund.pct_change(fill_method=None)

            cumulative = (1 + performance_pct.fillna(0)).cumprod() * 100
            drawdown = (cumulative - cumulative.cummax()) / cumulative.cummax()

            n_years = max((performance_fund.index[-1] - performance_fund.index[0]).days / 365.25, 1e-9)
            total_return = performance_fund.apply(lambda s: s.dropna().iloc[-1] / s.dropna().iloc[0] - 1)

            metrics = pd.DataFrame(index=performance_fund.columns)
            metrics['Returns'] = total_return
            metrics['Annualized Return'] = (1 + total_return) ** (1 / n_years) - 1
            metrics['Volatility'] = performance_pct.std() * np.sqrt(252)
            metrics['Sharpe Ratio'] = metrics['Annualized Return'] / metrics['Volatility']
            metrics['Drawdown'] = drawdown.min()
            metrics['Date Drawdown'] = drawdown.idxmin().dt.date
            if BENCHMARK_LABEL in performance_pct.columns:
                excess = performance_pct.drop(columns=BENCHMARK_LABEL).sub(performance_pct[BENCHMARK_LABEL], axis=0)
                metrics[f'Tracking Error to {BENCHMARK_LABEL}'] = excess.std() * np.sqrt(252)
            excess_core = performance_pct.drop(columns='Core').sub(performance_pct['Core'], axis=0)
            metrics['Tracking Error to Core'] = excess_core.std() * np.sqrt(252)
            display(display_scrollable_df(metrics.fillna(0).T.round(4)))

        updated_cumulative_perf(None)
        show_graph(None)
        get_holdings(None)

    optimize_btn.on_click(on_optimize_clicked)
    results_button = widgets.Button(description='Get Results', button_style='info')
    results_button.on_click(get_result)

    # =========================================================================
    # POSITIONING -- Current Portfolio.xlsx valued at the latest prices, with
    # average-cost P&L from Trade History.xlsx
    # =========================================================================
    positions_output = widgets.Output()
    holding_output = widgets.Output()

    def implied_prices():
        """Prices implied by the history files (value / units) -- offline fallback."""
        if positions.empty or quantities_holding.empty:
            return pd.DataFrame()
        cols = [c for c in quantities_holding.columns if c in positions.columns and c not in CASH_COLUMNS]
        q = quantities_holding[cols].reindex(positions.index)
        return (positions[cols] / q.where(q.abs() > 1e-12)).ffill()

    def fx_rates(currencies, start):
        """Daily BASE_CURRENCY per 1 unit of each currency (Yahoo_API.get_fx_rates).
        Currencies without a rate are treated as 1.0 (with a warning)."""
        fx = get_fx_rates(list(currencies), BASE_CURRENCY, start=start)
        for c in fx.attrs.get('missing', []):
            print(f"⚠️ No FX rate for {c} -> {BASE_CURRENCY}; treated as 1.0")
            fx[c] = 1.0
        return fx

    def spot_rates(fx, currencies):
        """Latest rate for each ticker's currency (1.0 when unknown)."""
        last = fx.ffill().iloc[-1]
        return currencies.map(lambda c: last.get(c, np.nan)).astype(float).fillna(1.0)

    def latest_prices(assets, rates):
        """Last price per asset in its quote currency: loaded universe ->
        yfinance -> history files (Positions / Quantities, converted from
        BASE_CURRENCY with the quote `rates`)."""
        assets = [a for a in assets if a not in CASH_COLUMNS]
        out = pd.Series(np.nan, index=assets, dtype=float)
        if not prices_local.empty:
            out = out.fillna(prices_local.ffill().iloc[-1].reindex(assets))
        missing = list(out[out.isna()].index)
        if missing:
            fetched = get_price_threading(missing, datetime.date.today() - datetime.timedelta(days=20))
            if not fetched.empty:
                out = out.fillna(fetched.ffill().iloc[-1].reindex(assets))
        missing = list(out[out.isna()].index)
        if missing:
            implied = implied_prices()
            if not implied.empty:
                out = out.fillna(implied.iloc[-1].reindex(assets) / rates.reindex(assets))
        return out

    def get_holdings(_=None):
        global holding_tickers, current_weights

        if current_holdings.empty:
            for out in (positions_output, holding_output):
                with out:
                    out.clear_output(wait=True)
                    print("⚠️ Current portfolio not loaded -- click 'Reload Excel'.")
            return

        assets = [a for a in current_holdings.index if a not in CASH_COLUMNS]
        with holding_output:
            holding_output.clear_output(wait=True)
            quote_ccy = quote_currencies(current_holdings.index, trades)
            trade_ccy = ticker_currencies(trades, current_holdings.index, quote=quote_ccy)
            fx = fx_rates(set(quote_ccy.values) | set(trade_ccy.values),
                          datetime.date.today() - datetime.timedelta(days=20))
            rates = spot_rates(fx, quote_ccy)
            cost_rates = spot_rates(fx, trade_ccy)
            px_last = latest_prices(assets, rates)

        # Price in the quote currency, Average Cost in the trade currency;
        # every money column from 'Market Value' on is in BASE_CURRENCY.
        holdings = pd.DataFrame({'Currency': quote_ccy, 'Quantity': current_holdings})
        holdings['Price'] = px_last.reindex(holdings.index)
        holdings.loc[holdings.index.isin(CASH_COLUMNS), 'Price'] = 1.0
        holdings['FX Rate'] = rates
        holdings['Market Value'] = holdings['Quantity'] * holdings['Price'] * holdings['FX Rate']
        amount = holdings['Market Value'].sum()
        holdings['Weights'] = holdings['Market Value'] / amount if amount else 0.0

        holding_tickers = assets
        current_weights = holdings.loc[assets, ['Weights']].rename(columns={'Weights': 'Current Weights'}).round(4)
        current_positions = holdings[['Market Value', 'Weights']].rename(
            columns={'Market Value': 'Current Portfolio', 'Weights': 'Current Weights'})

        with positions_output:
            positions_output.clear_output(wait=True)
            if dataframe.empty or returns_to_use.empty:
                print("⚠️ Load Prices.")
            elif quantities.empty:
                print("⚠️ Load Model.")
            else:
                model_values = (quantities.iloc[-1] * dataframe.iloc[-1]).dropna()
                model = pd.DataFrame({'Weights Model': model_values / model_values.sum()})
                out_of_model = current_positions.loc[~current_positions.index.isin(model.index), 'Current Portfolio'].sum()
                model['Model (without out of Model Positions)'] = model['Weights Model'] * (amount - out_of_model)
                model['Model'] = model['Weights Model'] * amount

                portfolio = pd.concat(
                    [model[['Model', 'Model (without out of Model Positions)', 'Weights Model']], current_positions],
                    axis=1).fillna(0)
                portfolio['Spread'] = portfolio['Current Portfolio'] - portfolio['Model']
                portfolio = portfolio.loc[~(portfolio == 0).all(axis=1)].sort_values(by='Weights Model', ascending=False)
                portfolio.loc['Total'] = portfolio.sum(axis=0)
                display(Markdown("### Model vs. Current Portfolio"))
                display(display_scrollable_df(portfolio.round(4)))

        with holding_output:
            if trades.empty:
                display(Markdown("### Current Portfolio"))
                display(display_scrollable_df(holdings.round(4)))
                print("⚠️ Trade history not loaded -- P&L not computed.")
                return

            avg_now = book_cost.iloc[-1] if not book_cost.empty else pd.Series(dtype=float)
            real_now = realized_pnl['Realized PnL'] if not realized_pnl.empty else pd.Series(dtype=float)
            fees_now = ledger['fees_daily'].iloc[-1] if not ledger['fees_daily'].empty else pd.Series(dtype=float)
            real_now = real_now.reindex(real_now.index.union(fees_now.index)).fillna(0)
            pnl = holdings.copy()
            pnl.insert(pnl.columns.get_loc('Market Value') + 2, 'Cost Currency', trade_ccy)
            pnl['Average Cost'] = avg_now.reindex(pnl.index)
            pnl['Cost FX Rate'] = cost_rates
            # P&L from the trades only: units implied by the trade history
            pnl['Units (trades)'] = trade_net_quantities.reindex(pnl.index)
            pnl['Book Cost'] = pnl['Average Cost'] * pnl['Units (trades)'] * pnl['Cost FX Rate']
            pnl['Unrealized P&L'] = pnl['Units (trades)'] * pnl['Price'] * pnl['FX Rate'] - pnl['Book Cost']
            # closed positions still carry realized P&L
            closed = real_now.index.difference(pnl.index)
            if len(closed):
                closed_ccy = ticker_currencies(trades, closed)
                closed_rates = spot_rates(fx_rates(closed_ccy.values, datetime.date.today() - datetime.timedelta(days=20)), closed_ccy)
                for t in closed:
                    pnl.loc[t] = np.nan
                    pnl.loc[t, 'Cost Currency'] = closed_ccy[t]
                    pnl.loc[t, 'Cost FX Rate'] = closed_rates[t]
            # realized P&L and fees are booked in the trade currency, shown at today's rate
            pnl['Realized P&L'] = (real_now.reindex(pnl.index) * pnl['Cost FX Rate']).fillna(0.0)
            pnl['Fees'] = (-fees_now.reindex(pnl.index) * pnl['Cost FX Rate']).fillna(0.0)
            pnl.loc[pnl.index.isin(CASH_COLUMNS), ['Average Cost', 'Book Cost', 'Unrealized P&L']] = np.nan
            pnl['Total P&L'] = pnl['Unrealized P&L'].fillna(0) + pnl['Realized P&L'] + pnl['Fees']
            pnl = pnl.sort_values(by='Weights', ascending=False)

            total = pnl.sum(numeric_only=True, min_count=1)
            total[['Quantity', 'Units (trades)', 'Price', 'Average Cost', 'FX Rate', 'Cost FX Rate']] = np.nan
            invested = pnl['Book Cost'].sum()
            pnl['Total P&L %'] = pnl['Total P&L'] / invested * 100 if invested else 0.0
            total['Total P&L %'] = total['Total P&L'] / invested * 100 if invested else 0.0
            pnl.loc['Total'] = total

            display(Markdown(f"### Current Portfolio and P&L (weighted-average book cost, values in {BASE_CURRENCY})"))
            display(display_scrollable_df(pnl.round(4)))

            # Reconcile the snapshot with the trades (deposits/transfers show up here)
            gap = (current_holdings.reindex(assets).fillna(0)
                   - trade_net_quantities.reindex(assets).fillna(0))
            gap = gap[gap.abs() > 1e-6]
            if not gap.empty:
                print("⚠️ Units in Current Portfolio that the trade history doesn't explain "
                      "(transfers? missing trades?):")
                display(display_scrollable_df(gap.to_frame('Unexplained Units').round(6)))

            display(Markdown("### Trade History"))
            display(display_scrollable_df(trades.sort_values('Date', ascending=False).set_index('Date')))

    position_button = widgets.Button(description='Get Positions', button_style='info')
    position_button.on_click(get_holdings)
    reload_excel_button = widgets.Button(description='Reload Excel', button_style='info')
    update_files_button = widgets.Button(description='Update Files', button_style='success', icon='save',
                                         tooltip="Save today's holdings into Quantities.xlsx and Positions.xlsx, "
                                                 "filling any days missed since the last save")
    update_files_output = widgets.Output()

    # --- layout ---
    overlay_ui = widgets.VBox([widgets.HBox([widgets.VBox([strat_overlay, dropdown_strategy_limit]),
                                             widgets.VBox([add_strategy_btn, clear_strategy_btn])]),
                               overlay_output])
    risk_strat_ui = widgets.VBox([widgets.HBox([widgets.VBox([risk_type, dropdown_risk_sign, risk_limit]),
                                                widgets.VBox([add_risk_btn, clear_risk_btn])]),
                                  risk_constraints_output])
    allocation_ui = widgets.VBox([
        widgets.HBox([widgets.VBox([dropdown_asset, dropdown_sign, dropdown_limit]),
                      widgets.VBox([add_constraint_btn, clear_constraints_btn, optimize_btn])]),
        constraint_output, grid, overlay_ui, risk_strat_ui,
        widgets.HBox([button_add, button_clear, results_button])])
    constraint_ui = widgets.VBox([
        widgets.HBox([start_date_perf, end_date_perf, refresh_perf_button]),
        main_output,
        widgets.VBox([strat, rebalancing_frequency, benchmark_tracking_error, window_vol]),
        allocation_ui, strategy_output,
        widgets.HBox([start_date_perf, end_date_perf, refresh_perf_button]),
        output_returns, widgets.HBox([perf_output, drawdown_output]), widgets.HBox([vol_output, frontier_output])])
    def on_currency_settings(_=None):
        """Switch base currency / conversion and rebuild whatever is already loaded."""
        global BASE_CURRENCY, CONVERT_TO_BASE, positions
        BASE_CURRENCY = base_currency_dd.value
        CONVERT_TO_BASE = convert_checkbox.value
        positions = positions_in_base()
        if not dataframe.empty:
            get_prices()
        get_holdings()
        if not historical_ptf.empty:
            get_ex_post_returns(None)

    base_currency_dd.observe(on_currency_settings, names='value')
    convert_checkbox.observe(on_currency_settings, names='value')

    universe_ui = widgets.VBox([
        search_ui,
        widgets.HBox([base_currency_dd, convert_checkbox]),
        widgets.HBox([start_date, data_button]),
        scope_output,
        widgets.HBox([start_date_perf, end_date_perf, refresh_perf_button]),
        main_output, price_output])
    calendar_perf = widgets.VBox([widgets.HBox([frequency_graph, fund, benchmark, graph_button]), calendar_output])
    positions_ui = widgets.VBox([widgets.HBox([position_button, reload_excel_button, update_files_button]),
                                 update_files_output, positions_output, holding_output])
    rebalancing_frequency_pnl = widgets.Dropdown(description='Frequency:', options=['Yearly', 'Quarterly', 'Monthly'], value='Monthly')

    # =========================================================================
    # Shared helpers for ex-post weights / underlying returns
    # =========================================================================
    def weights_ex_post_frame():
        """Daily weights from Positions.xlsx: value / Total, cash columns removed."""
        if positions.empty:
            return pd.DataFrame()
        pos = positions.drop(columns=[c for c in CASH_COLUMNS if c in positions.columns])
        w = pos.drop(columns='Total').div(positions['Total'], axis=0)
        w = w.replace([np.inf, -np.inf], np.nan).fillna(0.0).sort_index()
        return w[~w.index.duplicated(keep='last')]

    def fetch_underlying_returns(weights):
        """Simple returns of every model + held ticker since the first position date."""
        combined = set(weights.columns) | set(quantities.columns)
        start = weights.index[0].date()
        px_, _ = prices_to_base(get_price_threading(combined, start), start)
        return px_.pct_change(fill_method=None)

    def resolve_weight_series(series, name):
        if name in series:
            return series[name]
        if 'Rebalanced ' + name in series:
            return series['Rebalanced ' + name]
        return None

    def build_weight_series_dict(start_ts, end_ts):
        """Weight history per allocation / model, shared by the risk tabs."""
        series = {}
        # Buy & Hold / Rebalanced always start on the backtest's first day (the
        # portfolio you'd actually have held); each chart then shows its own
        # date range of that one path, drift included. Frequency passed
        # explicitly (the helper's default is Quarterly).
        for key in grid.data.index:
            w = grid.data.loc[key].reindex(dataframe.columns).fillna(0)
            rebalanced_series = rebalanced_portfolio(dataframe, w, frequency=rebalancing_frequency_pnl.value)
            buy_and_hold_series = buy_and_hold(dataframe, w)
            series['Rebalanced ' + key] = rebalanced_series.div(rebalanced_series.sum(axis=1), axis=0).loc[start_ts:end_ts]
            series['Buy and Hold ' + key] = buy_and_hold_series.div(buy_and_hold_series.sum(axis=1), axis=0).loc[start_ts:end_ts]
        for name, q in (('Fund', quantities), ('Core', quantities_core), ('Overlay', quantities_overlay)):
            if not q.empty:
                ptf = q * dataframe
                series[name] = ptf.div(ptf.sum(axis=1), axis=0).loc[start_ts:end_ts]
        if BENCHMARK_TICKER in dataframe.columns:
            bench = pd.DataFrame(0.0, index=dataframe.index, columns=dataframe.columns)
            bench[BENCHMARK_TICKER] = 1.0
            series[BENCHMARK_LABEL] = bench.loc[start_ts:end_ts]
        return series

    # =========================================================================
    # RISK ANALYSIS TAB -- ex-ante risk contribution
    # =========================================================================
    risk_output = widgets.Output()
    ex_ante_output = widgets.Output()
    start_date_perf_risk = widgets.DatePicker(value=start_perf_date, layout=widgets.Layout(width='350px'))
    end_date_perf_risk = widgets.DatePicker(value=datetime.date.today(), layout=widgets.Layout(width='350px'))

    def risk_range_returns(out):
        """Validates inputs for the risk tabs, returns (prices, returns) or None."""
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        msg = None
        if rng is None:
            msg = "⚠️ Invalid date range."
        elif dataframe.empty or returns_to_use.empty or grid.data.empty:
            msg = "⚠️ Please compute optimization results first."
        if msg is None:
            range_prices = dataframe.loc[rng[0]:rng[1]]
            range_returns = range_prices.pct_change(fill_method=None).dropna()
            if range_returns.empty:
                msg = "⚠️ No data available in selected date range."
        if msg:
            with out:
                out.clear_output()
                print(msg)
            return None
        return range_prices, range_returns

    def update_fund_display(_=None):
        data = risk_range_returns(risk_output)
        if data is None:
            return
        range_prices, range_returns = data
        if selected_fund.value not in grid.data.index or selected_bench.value not in grid.data.index:
            with risk_output:
                risk_output.clear_output()
                print("⚠️ Select a fund and a benchmark from the allocation grid.")
            return

        cols = range_returns.columns
        selected_weights = grid.data.loc[selected_fund.value].reindex(cols).fillna(0)
        bench_weights = grid.data.loc[selected_bench.value].reindex(cols).fillna(0)
        active_weights = selected_weights - bench_weights

        portfolio = RiskAnalysis(range_returns)
        decomposition = pd.DataFrame(portfolio.var_contrib(selected_weights)[0]) * 100
        te_decomposition = pd.DataFrame(portfolio.var_contrib(active_weights)[0]) * 100
        bench_returns = range_returns.dot(bench_weights)
        beta_asset = compute_per_asset_betas(range_returns, bench_returns)
        beta_decomposition = pd.Series(beta_asset * selected_weights.to_numpy(), index=cols).to_frame('Beta Contribution')

        def simulated_pnl(weights):
            # holdings started on the backtest's first day, P&L measured over this window
            q_reb = (rebalanced_portfolio(dataframe, weights, frequency=rebalancing_frequency_pnl.value) / dataframe).loc[range_prices.index]
            q_bh = (buy_and_hold(dataframe, weights) / dataframe).loc[range_prices.index]
            pnl_bh = (q_bh * range_prices - rebalanced_book_cost(range_prices, q_bh)).iloc[-1]
            pnl_reb = (q_reb * range_prices - rebalanced_book_cost(range_prices, q_reb)).iloc[-1]
            return pd.DataFrame({'Profit and Loss (Buy and Hold)': pnl_bh,
                                 'Profit and Loss (Rebalanced)': pnl_reb})

        fund_table = pd.concat([simulated_pnl(selected_weights), decomposition], axis=1)
        fund_table.loc['Total'] = fund_table.sum(axis=0)
        active_table = pd.concat([simulated_pnl(active_weights), te_decomposition, beta_decomposition], axis=1)
        active_table.loc['Total'] = active_table.sum(axis=0)

        def sort_table(t):
            t = t.fillna(0)
            return t.sort_values(by='Vol Contribution', ascending=False) if 'Vol Contribution' in t.columns else t

        with risk_output:
            risk_output.clear_output(wait=True)
            display(Markdown("### Performance and Risk Contribution"))
            display(display_scrollable_df(sort_table(fund_table)))
            display(Markdown("### Excess P&L and TE Contribution"))
            display(display_scrollable_df(sort_table(active_table)))

    def ex_ante_metrics(bench_name):
        data = risk_range_returns(ex_ante_output)
        if data is None:
            return
        _, range_returns = data
        if bench_name not in grid.data.index:
            with ex_ante_output:
                ex_ante_output.clear_output()
                print("⚠️ Select a benchmark from the allocation grid.")
            return

        cols = range_returns.columns
        portfolio = RiskAnalysis(range_returns)
        bench_w = grid.data.loc[bench_name].reindex(cols).fillna(0)
        beta = compute_per_asset_betas(range_returns, range_returns.dot(bench_w))

        rows = {}
        for name in grid.data.index:
            w = grid.data.loc[name].reindex(cols).fillna(0)
            rows[name] = {'Vol Ex Ante': portfolio.variance(w),
                          'Tracking Error Ex Ante': portfolio.variance(w - bench_w),
                          'Beta Ex Ante': float(np.dot(w.to_numpy(), beta))}
        with ex_ante_output:
            ex_ante_output.clear_output(wait=True)
            display(Markdown("### Ex Ante Metrics"))
            display(display_scrollable_df(pd.DataFrame(rows).T))

    def on_fund_change(change):
        if change['new'] in grid.data.index:
            update_fund_display()

    def on_bench_change(change):
        if change['new'] in grid.data.index:
            ex_ante_metrics(change['new'])
            update_fund_display()

    def update_contrib_and_ex_ante(_):
        update_fund_display()
        ex_ante_metrics(selected_bench.value)

    selected_fund.observe(on_fund_change, names='value')
    selected_bench.observe(on_bench_change, names='value')
    rebalancing_frequency_pnl.observe(lambda ch: update_fund_display(), names='value')
    start_date_perf_risk.observe(update_contrib_and_ex_ante, names='value')
    end_date_perf_risk.observe(update_contrib_and_ex_ante, names='value')

    # =========================================================================
    # RISK ANALYSIS TAB -- VaR / CVaR simulation
    # =========================================================================
    var_output = widgets.Output()
    stress_grid_output = widgets.Output()
    var_scenarios, cvar_scenarios, fund_results = {}, {}, {}
    stress_factor = widgets.BoundedFloatText(value=1.0, min=1.0, max=3.0, step=0.1, description='Stress Factor:')
    mean_factor = widgets.BoundedFloatText(value=1.0, min=0.0, max=3.0, step=0.1, description='Mean Factor:')
    iterations = widgets.BoundedIntText(value=10000, min=1000, max=100000, step=1, description='Iterations:')
    num_scenarios = widgets.BoundedIntText(value=100, min=1, max=1000, step=1, description='Scenarios:')
    var_centile = widgets.BoundedFloatText(value=0.05, min=0, max=1, step=0.01, description='VaR Centile:')
    loading_bar_var = widgets.IntProgress(description='Loading scenarios...', min=0, max=100, style={'description_width': '150px'})

    grid_stress = DataGrid(pd.DataFrame(), editable=True, layout={"height": "250px"})
    grid_mean_shock = DataGrid(pd.DataFrame(), editable=True, layout={"height": "250px"})

    def update_stress_grid(_cell=None):
        new_matrix = set_symmetric(grid_stress.data.to_numpy(), limit=2)
        grid_stress.data = pd.DataFrame(new_matrix, columns=dataframe.columns, index=dataframe.columns)
        update_stress_data(None)

    def update_stress_data(_=None):
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        if rng is None or dataframe.empty or grid_stress.data.empty:
            with stress_grid_output:
                stress_grid_output.clear_output()
                print("⚠️ Load prices and choose a valid date range.")
            return
        range_returns = dataframe.loc[rng[0]:rng[1]].pct_change(fill_method=None)
        cov = range_returns.cov().to_numpy()

        shock = grid_stress.data.to_numpy()
        stress_matrix = np.diag(np.diag(shock))
        stressed_cov = stress_matrix @ cov @ stress_matrix
        stressed_std = np.sqrt(np.diag(stressed_cov))
        shocked_means = range_returns.mean() * grid_mean_shock.data['Mean Shock'] * 250

        off_diag = np.tril(shock, -1)
        corr_matrix = stressed_cov / np.outer(stressed_std, stressed_std) + off_diag + off_diag.T
        corr_matrix = cov_nearest(np.clip(corr_matrix, -1, 1))
        corr_dataframe = pd.DataFrame(corr_matrix, index=range_returns.columns, columns=range_returns.columns)

        expected_data = pd.DataFrame({'Means': range_returns.mean() * 250,
                                      'Shocked Means': shocked_means,
                                      'Vol': range_returns.std() * np.sqrt(250),
                                      'Shocked Vol': pd.Series(stressed_std * np.sqrt(250), index=range_returns.columns)})

        with stress_grid_output:
            stress_grid_output.clear_output(wait=True)
            vol_out, mean_out, exp_out, corr_old, corr_new = (widgets.Output() for _ in range(5))
            with vol_out:
                display(Markdown('## Correlation Shock'))
                display(grid_stress)
            with mean_out:
                display(Markdown('## Returns Shock'))
                display(grid_mean_shock)
            with exp_out:
                display(Markdown('## Mean and Vol Changes'))
                display(display_scrollable_df(expected_data))
            with corr_old:
                display(Markdown('### Original Correlation'))
                display(display_scrollable_df(range_returns.corr()))
            with corr_new:
                display(Markdown('### New Correlation'))
                display(display_scrollable_df(corr_dataframe))
            display(widgets.VBox([vol_out, mean_out, exp_out]))
            display(Markdown('## Correlation Changes'))
            display(widgets.HBox([corr_old, corr_new]))

    def reset_stress(_=None):
        global grid_stress, grid_mean_shock
        if returns_to_use.empty:
            with stress_grid_output:
                stress_grid_output.clear_output()
                print("⚠️ Load Prices")
            return
        n = dataframe.shape[1]
        grid_stress = DataGrid(
            pd.DataFrame(np.diag(np.full(n, stress_factor.value)), columns=dataframe.columns, index=dataframe.columns),
            editable=True, layout={"height": "250px"})
        grid_mean_shock = DataGrid(
            pd.DataFrame(np.full(n, mean_factor.value), columns=['Mean Shock'], index=dataframe.columns),
            editable=True, layout={"height": "250px"})
        # Cell edits only -- setting .data programmatically doesn't re-fire these.
        grid_stress.on_cell_change(update_stress_grid)
        grid_mean_shock.on_cell_change(update_stress_data)
        update_stress_data(None)

    stress_grid_button = widgets.Button(description="Refresh")
    stress_grid_button.on_click(reset_stress)

    def get_var_metrics(_):
        global var_scenarios, cvar_scenarios, fund_results

        with var_output:
            var_output.clear_output()
            rng = parse_range(start_date_perf_risk, end_date_perf_risk)
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if dataframe.empty or returns_to_use.empty or grid.data.empty:
                print("⚠️ Please compute optimization results first.")
                return
            if grid_stress.data.empty:
                print("⚠️ Click 'Refresh' to build the stress grids first.")
                return

            horizon = 1 / 250
            spot = dataframe.iloc[-1]
            theta = 2
            stress_matrix = grid_stress.data.to_numpy()
            mean_shock_vec = grid_mean_shock.data['Mean Shock']
            distrib_functions = {
                'multivariate_distribution': (iterations.value, stress_matrix, mean_shock_vec),
                'gaussian_copula': (iterations.value, stress_matrix, mean_shock_vec),
                't_copula': (iterations.value, stress_matrix, mean_shock_vec),
                'gumbel_copula': (iterations.value, theta, stress_factor.value, mean_shock_vec),
                'monte_carlo': (spot, horizon, iterations.value, stress_matrix, mean_shock_vec)}

            range_returns = dataframe.loc[rng[0]:rng[1]].pct_change(fill_method=None)
            portfolio = RiskAnalysis(range_returns)
            var_scenarios, cvar_scenarios, fund_results = {}, {}, {}
            loading_bar_var.value = 0
            display(loading_bar_var)

            def process_index(index):
                vs, cvs = {}, {}
                for func_name, args in distrib_functions.items():
                    func = getattr(portfolio, func_name)
                    scenarios = {}
                    for i in range(num_scenarios.value):
                        raw = func(*args)
                        if func_name == 'monte_carlo':
                            raw = raw[1]
                        distrib = pd.DataFrame(raw, columns=portfolio.returns.columns)
                        w = grid.data.loc[index].reindex(distrib.columns).fillna(0)
                        distrib = (distrib * w).loc[:, w > 0]
                        distrib['Portfolio'] = distrib.sum(axis=1)
                        scenarios[i] = distrib.sort_values(by='Portfolio').iloc[int(distrib.shape[0] * var_centile.value)]
                    scenario = pd.DataFrame(scenarios).T
                    mean_scenario = scenario.mean()
                    cvar = scenario.loc[scenario['Portfolio'] < mean_scenario['Portfolio']].mean()
                    vs[func_name] = mean_scenario
                    cvs[func_name] = cvar
                return index, vs, cvs, {'Value At Risk': mean_scenario.loc['Portfolio'], 'CVaR': cvar.loc['Portfolio']}

            n_funds = len(grid.data.index)
            with ThreadPoolExecutor() as executor:
                futures = [executor.submit(process_index, idx) for idx in grid.data.index]
                for future in as_completed(futures):
                    idx, vs, cvs, fund_result = future.result()
                    var_scenarios[idx] = vs
                    cvar_scenarios[idx] = cvs
                    fund_results[idx] = fund_result
                    loading_bar_var.value += int(100 / n_funds)
            loading_bar_var.value = 0

        display_var_results(selected_fund_var.value)

    var_function_names = {'Multivariate': 'multivariate_distribution',
                          'Gaussian Copula': 'gaussian_copula',
                          'Monte Carlo': 'monte_carlo',
                          'Gumbel Copula': 'gumbel_copula',
                          'T-Copula': 't_copula'}
    result_var = pd.DataFrame()
    result_cvar = pd.DataFrame()
    series_dict = {}
    current_underlying_returns = pd.DataFrame()

    value_at_risk_trajectory_output = widgets.Output()
    selected_fund_to_decompose_var = widgets.Dropdown(options=[], description='Fund:')
    var_risk_trajectory_refresh_button = widgets.Button(description='Refresh')
    var_trajectory_button = widgets.Button(description='Get VaR', button_style='success')
    func_name = widgets.Dropdown(description='Method', options=list(var_function_names.keys()), value='Gumbel Copula')
    window_var = widgets.IntText(value=252, description='Window:')
    loading_bar_risk = widgets.IntProgress(description='Loading...', min=0, max=100, style={'description_width': '150px'})

    def trajectory_distrib_args(method):
        spot = dataframe.iloc[-1]
        theta = 2
        distrib_functions = {
            'multivariate_distribution': (iterations.value, stress_factor.value, mean_factor.value),
            'gaussian_copula': (iterations.value, stress_factor.value, mean_factor.value),
            't_copula': (iterations.value, stress_factor.value, mean_factor.value),
            'gumbel_copula': (iterations.value, theta, stress_factor.value, mean_factor.value),
            'monte_carlo': (spot, 1 / 250, iterations.value, stress_factor.value, mean_factor.value)}
        return distrib_functions[method]

    def value_at_risk_trajectory(_):
        global result_var, result_cvar, series_dict, current_underlying_returns

        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        with value_at_risk_trajectory_output:
            value_at_risk_trajectory_output.clear_output(wait=True)
            if rng is None:
                print("⚠️ Invalid start or end date.")
                return
            if dataframe.empty or returns_to_use.empty or grid.data.empty:
                print("⚠️ Please compute optimization results first.")
                return
            if positions.empty:
                print("⚠️ Positions not loaded -- click 'Reload Excel'.")
                return
            start_ts, end_ts = rng
            if len(returns_to_use.loc[start_ts:end_ts]) < window_var.value:
                print("⚠️ Date range is shorter than rolling window.")
                return
            display(loading_bar_risk)

            series_dict = {}
            for key in grid.data.index:
                w = grid.data.loc[key].reindex(dataframe.columns).fillna(0)
                # started on the backtest's first day, then cut to this window
                reb = rebalanced_portfolio(dataframe, w, frequency=rebalancing_frequency_pnl.value)
                bh = buy_and_hold(dataframe, w)
                series_dict['Rebalanced ' + key] = reb.div(reb.sum(axis=1), axis=0).loc[start_ts:end_ts]
                series_dict['Buy and Hold ' + key] = bh.div(bh.sum(axis=1), axis=0).loc[start_ts:end_ts]
            for name, q in (('Fund', quantities), ('Core', quantities_core), ('Overlay', quantities_overlay)):
                if not q.empty:
                    ptf = q * dataframe
                    series_dict[name] = ptf.div(ptf.sum(axis=1), axis=0).loc[start_ts:end_ts]

            weights_ex_post = weights_ex_post_frame()
            current_underlying_returns = fetch_underlying_returns(weights_ex_post)

            method = var_function_names[func_name.value]
            args = trajectory_distrib_args(method)
            tasks = [(key, returns_to_use.loc[start_ts:end_ts], series_dict[key]) for key in series_dict]
            series_dict['Historical Portfolio'] = weights_ex_post.loc[start_ts:end_ts]
            if method == 'gumbel_copula':
                w_hist = weights_ex_post.loc[start_ts:end_ts]
                tasks.append(('Historical Portfolio',
                              current_underlying_returns.reindex(w_hist.index), w_hist))

            results_dict_var, results_dict_cvar = {}, {}
            loading_bar_risk.value = 0
            loading_bar_risk.max = len(tasks)
            for name, returns, weight in tasks:
                common_col = returns.columns.intersection(weight.columns)
                common_index = returns.index.intersection(weight.index)
                try:
                    var_data, cvar_data = get_var_contribution(
                        method, args, returns.loc[common_index, common_col],
                        weight.loc[common_index, common_col], window_var.value, var_centile.value)
                    results_dict_var[name] = var_data['Portfolio']
                    results_dict_cvar[name] = cvar_data['Portfolio']
                except Exception as e:
                    print(f"VaR failed for {name}: {e}")
                loading_bar_risk.value += 1

            if not results_dict_var:
                print("⚠️ No risk results were computed.")
                return
            result_var = pd.concat(results_dict_var, axis=1)
            result_cvar = pd.concat(results_dict_cvar, axis=1)

        options = list(result_var.columns)
        set_choices(selected_fund_to_decompose_var, options, 'Fund')
        show_var_graph(None)

    var_trajectory_button.on_click(value_at_risk_trajectory)

    def show_var_graph(_):
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        with value_at_risk_trajectory_output:
            value_at_risk_trajectory_output.clear_output(wait=True)
            if rng is None:
                print("⚠️ Invalid start or end date.")
                return
            if result_var.empty or result_cvar.empty or not series_dict:
                print("⚠️ Click 'Get VaR' first.")
                return
            start_ts, end_ts = rng
            name = selected_fund_to_decompose_var.value
            series_weights = series_dict.get(name)
            if series_weights is None:
                print(f"⚠️ No weights for '{name}'.")
                return
            returns_src = current_underlying_returns if name == 'Historical Portfolio' else returns_to_use
            common = series_weights.columns.intersection(returns_src.columns)
            common_index = series_weights.index.intersection(returns_src.index)
            method = var_function_names[func_name.value]
            var, cvar = get_var_contribution(
                method, trajectory_distrib_args(method),
                returns_src.loc[common_index, common].loc[start_ts:end_ts],
                series_weights.loc[common_index, common].loc[start_ts:end_ts],
                window_var.value, var_centile.value)

            left, right = widgets.Output(), widgets.Output()
            with left:
                fig = px.line(result_var.loc[start_ts:end_ts], title='Portfolios Value At Risk', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig, pct_axis=True), [name] if name in result_var.columns else ["Historical Portfolio", "Fund"]).show()
                fig4 = px.line(var, title='Value at Risk History', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig4, pct_axis=True), ["Portfolio"]).show()
            with right:
                fig2 = px.line(result_cvar.loc[start_ts:end_ts], title='Portfolio Expected Shortfall', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig2, pct_axis=True), [name] if name in result_cvar.columns else ["Historical Portfolio", "Fund"]).show()
                fig3 = px.line(cvar, title='Expected Shortfall History', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig3, pct_axis=True), ["Portfolio"]).show()
            display(widgets.HBox([left, right]))

    var_risk_trajectory_refresh_button.on_click(show_var_graph)

    def display_var_results(fund_name):
        with var_output:
            var_output.clear_output(wait=True)
            if fund_name not in var_scenarios:
                print(f"⚠️ No VaR data found for '{fund_name}'. Run simulation first.")
                return
            columns = ['Multivariate', 'Gaussian Copula', 'T-Student Copula', 'Gumbel Copula', 'Monte Carlo']
            var_dataframe = pd.DataFrame(var_scenarios[fund_name])
            var_dataframe.columns = columns
            cvar_dataframe = pd.DataFrame(cvar_scenarios[fund_name])
            cvar_dataframe.columns = columns
            display(Markdown("### Summary"))
            display(display_scrollable_df(pd.DataFrame(fund_results).T))
            display(Markdown(f"### VaR Results for **{fund_name}**"))
            display(display_scrollable_df(var_dataframe))
            display(Markdown(f"### CVaR Results for **{fund_name}**"))
            display(display_scrollable_df(cvar_dataframe))

    selected_fund_var.observe(lambda ch: display_var_results(ch['new']) if ch['new'] in var_scenarios else None, names='value')
    get_var_button = widgets.Button(description='Run Simulation', button_style='info')
    get_var_button.on_click(get_var_metrics)

    ex_ante_ui = widgets.VBox([widgets.HBox([start_date_perf_risk, end_date_perf_risk]),
                               widgets.VBox([selected_fund, selected_bench, rebalancing_frequency_pnl]),
                               risk_output, ex_ante_output])
    var_ui = widgets.VBox([widgets.HBox([start_date_perf_risk, end_date_perf_risk]),
                           widgets.VBox([selected_fund_var, stress_factor, mean_factor, iterations, num_scenarios, var_centile,
                                         stress_grid_output, widgets.HBox([get_var_button, stress_grid_button])]),
                           var_output])
    var_history_ui = widgets.VBox([widgets.HBox([start_date_perf_risk, end_date_perf_risk, var_trajectory_button, var_risk_trajectory_refresh_button]),
                                   selected_fund_to_decompose_var, func_name, window_var, var_centile, value_at_risk_trajectory_output])

    # =========================================================================
    # MARKET RISK TAB -- PCA factor decomposition
    # =========================================================================
    pca_output = widgets.Output()
    pca_components = widgets.Output()
    start_date_market_risk = widgets.DatePicker(value=start_perf_date, layout=widgets.Layout(width='350px'))
    end_date_market_risk = widgets.DatePicker(value=datetime.date.today(), layout=widgets.Layout(width='350px'))
    quantities_eigen = pd.DataFrame()
    perf_index_eigen = pd.DataFrame()

    selected_components = widgets.Dropdown(options=['PC1'], description='Select PCA:', style={'description_width': '150px'})
    num_components = widgets.BoundedIntText(min=1, max=5, value=5, description='PCA Components:', style={'description_width': '150px'})
    num_closest_to_pca = widgets.BoundedIntText(min=1, max=20, value=5, description='PCA Closest:', style={'description_width': '150px'})

    def get_market_risk_metrics(_):
        rng = parse_range(start_date_market_risk, end_date_market_risk)
        pca_output.clear_output()
        with pca_components:
            pca_components.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if returns_to_use.empty:
                print('⚠️ Load Prices.')
                return

        market_tickers = [t for t in tickers if t in dataframe.columns]
        range_returns = returns_to_use.loc[rng[0]:rng[1], market_tickers].dropna()
        num_components.max = len(range_returns.columns)
        num_closest_to_pca.max = len(range_returns.columns)
        portfolio = RiskAnalysis(range_returns)
        eigval, eigvec, portfolio_components = portfolio.pca(num_components=min(num_components.value, len(range_returns.columns)))
        current = selected_components.value
        set_choices(selected_components, list(portfolio_components.columns), current)
        comp = selected_components.value

        variance_explained = pd.DataFrame(eigval / eigval.sum(), index=portfolio_components.columns, columns=['Variance Explained'])
        pca_weight = portfolio_components[comp] / portfolio_components[comp].sum()
        pca_portfolio = portfolio_components[[comp]].sort_values(by=comp, ascending=False)

        historical_PCA = pd.DataFrame(portfolio.returns.to_numpy() @ pca_weight.to_numpy(),
                                      index=portfolio.returns.index, columns=['PCA']).dropna()
        historical_PCA.iloc[0] = 0

        comparison = portfolio.returns.copy()
        comparison['PCA'] = historical_PCA['PCA']
        distances = np.sqrt(((comparison.sub(historical_PCA['PCA'], axis=0)) ** 2).sum()).sort_values()
        pca_similarity = comparison[distances.index[:num_closest_to_pca.value]].copy()
        pca_similarity.iloc[0] = 0
        pca_similarity = (1 + pca_similarity).cumprod() * 100

        with pca_components:
            pca_components.clear_output(wait=True)
            dark_layout(px.bar(variance_explained, title='Variance Explanation in %'), pct_axis=True, width=800, height=400).show()
            dark_layout(px.bar(pca_portfolio, title='Eigen Weights'), pct_axis=True, width=800, height=400).show()
        with pca_output:
            pca_output.clear_output(wait=True)
            dark_layout(px.line((1 + historical_PCA).cumprod() * 100, title='Eigen Index', render_mode='svg'), width=800, height=400).show()
            dark_layout(px.line(pca_similarity, title='PCA Similarity', render_mode='svg'), width=800, height=400).show()

    asset_output_corr = widgets.Output()
    window_corr = widgets.IntText(value=252, description='Rolling Correlation:', style={'description_width': '150px'})

    def update_correlation(_=None):
        rng = parse_range(start_date_market_risk, end_date_market_risk)
        with asset_output_corr:
            asset_output_corr.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if returns_to_use.empty:
                print('⚠️ Load Prices.')
                return
            if dropdown_asset1.value == dropdown_asset2.value:
                print('⚠️ Same asset')
                return

        range_returns = returns_to_use.loc[rng[0]:rng[1]]
        pca_over_time = first_pca_over_time(returns=range_returns, window=window_corr.value)
        a1, a2 = dropdown_asset1.value, dropdown_asset2.value
        rolling_correlation = range_returns[a1].rolling(window_corr.value).corr(range_returns[a2]).dropna()
        rolling_mean_returns = range_returns.rolling(window_corr.value).mean().dropna() * 252

        with asset_output_corr:
            asset_output_corr.clear_output(wait=True)
            o1, o2, o3, o4 = (widgets.Output() for _ in range(4))
            with o1:
                dark_layout(px.line(rolling_correlation, title=f"{a1}/{a2} Correlation", render_mode='svg'), width=800, height=400).show()
            with o2:
                fig2 = px.imshow(range_returns.corr().round(2), title='Correlation Matrix', color_continuous_scale='Blues', text_auto=True, aspect="auto")
                fig2.update_traces(xgap=2, ygap=2)
                dark_layout(fig2, width=800, height=400).show()
            with o3:
                fig3 = px.line(pca_over_time, title='First principal component (Variance Explained in %)', render_mode='svg')
                dark_layout(fig3, pct_axis=True, width=800, height=400, xaxis_title=None, yaxis_title=None).show()
            with o4:
                fig4 = px.line(rolling_mean_returns, title='Mean Return', render_mode='svg')
                only_show(dark_layout(fig4, pct_axis=True, width=800, height=400, xaxis_title=None, yaxis_title=None), [a1, a2]).show()
            display(widgets.HBox([widgets.VBox([o1, o4]), widgets.VBox([o3, o2])]))

    market_button = widgets.Button(description='Market Risk Analysis', button_style='info')
    market_button.on_click(get_market_risk_metrics)
    correlation_button = widgets.Button(description='Get Correlation', button_style='info')
    correlation_button.on_click(update_correlation)
    market_ui = widgets.VBox([widgets.HBox([start_date_market_risk, end_date_market_risk, market_button]),
                              num_components, selected_components, num_closest_to_pca,
                              widgets.HBox([pca_components, pca_output])])
    correlation_ui = widgets.VBox([widgets.HBox([start_date_market_risk, end_date_market_risk, correlation_button]),
                                   dropdown_asset1, dropdown_asset2, window_corr, asset_output_corr])

    frequency_eigen = widgets.Dropdown(description='Frequency:', options=['Yearly', 'Quarterly', 'Monthly'], value='Quarterly', style={'description_width': '150px'})
    selected_pca_market = widgets.Dropdown(description='PCA:', options=['PC1', 'PC2', 'PC3'], value='PC1', style={'description_width': '150px'})
    market_vol_window = widgets.IntText(value=252, description='Window:', style={'description_width': '150px'})
    market_factor_output = widgets.Output()
    market_factors_button = widgets.Button(description="Get Market Drivers", button_style="info")

    def show_market_graph(_):
        rng = parse_range(start_date_market_risk, end_date_market_risk)
        with market_factor_output:
            market_factor_output.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if returns_to_use.empty:
                print('⚠️ Load Prices.')
                return
            if quantities_eigen.empty:
                print('⚠️ Load Factors.')
                return

        start_ts, end_ts = rng
        range_prices = dataframe.loc[start_ts:end_ts]
        range_returns = returns_to_use.loc[start_ts:end_ts]
        q = quantities_eigen.reindex(range_prices.index).ffill()

        market_portfolio = q * range_prices
        weights_series = market_portfolio.div(market_portfolio.sum(axis=1), axis=0)

        perf = perf_index_eigen.loc[start_ts:end_ts].copy()
        if not perf.empty:
            perf.iloc[0] = 0
        market_results = (1 + perf).cumprod() * 100

        market_pnl = market_portfolio - rebalanced_book_cost(range_prices, q)
        market_pnl['Market Index'] = market_pnl.sum(axis=1)

        vol_contribution = get_ex_ante_vol_contribution(weights_series, range_returns, window=market_vol_window.value)
        correlation_contribution = get_correlation_contribution(weights_series, range_returns, window=market_vol_window.value)
        idiosyncratic_contribution = get_idiosyncratic_contribution(weights_series, range_returns, window=market_vol_window.value)

        with market_factor_output:
            market_factor_output.clear_output(wait=True)
            left, right = widgets.Output(), widgets.Output()
            with left:
                fig = px.line(market_results, title='Performance Comparison', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig), ["Market Index", "Fund", BENCHMARK_LABEL, "Historical Portfolio"]).show()
                fig2 = px.line(market_pnl, title='Market Drivers', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig2), ["Market Index"]).show()
                fig3 = px.line(correlation_contribution, title='Market Correlation', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig3, pct_axis=True), ["Total Correlation"]).show()
            with right:
                fig4 = px.line(vol_contribution, title='Market Volatility', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig4, pct_axis=True), ["Total Vol"]).show()
                fig5 = px.line(idiosyncratic_contribution, title='Market Intrinsic Volatility', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig5, pct_axis=True), ["Total Idiosyncratic Vol"]).show()
                fig6 = px.line(weights_series, title='Market Weights', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig6, pct_axis=True), [BENCHMARK_TICKER]).show()
            display(widgets.HBox([left, right]))

    def get_market_factors(_):
        global quantities_eigen, perf_index_eigen
        rng = parse_range(start_date_market_risk, end_date_market_risk)
        with market_factor_output:
            market_factor_output.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if returns_to_use.empty:
                print('⚠️ Load Prices.')
                return

        start_ts, end_ts = rng
        range_prices = dataframe.loc[start_ts:end_ts]
        range_returns = returns_to_use.loc[start_ts:end_ts]
        asset_cols = returns_to_use.columns

        dates = get_rebalancing_dates(range_returns, frequency=frequency_eigen.value)
        tasks = [(returns_to_use.loc[dates[i]:dates[i + 1]], dates[i], dates[i + 1]) for i in range(len(dates) - 1)]
        loading_bar_market = widgets.IntProgress(description='Loading Drivers...', min=0, max=max(len(tasks), 1), style={'description_width': '150px'})
        with market_factor_output:
            display(loading_bar_market)

        pc = selected_pca_market.value

        def worker(subset, start, end):
            if subset.empty or len(subset) < 2:
                return None
            try:
                _, _, portfolio_components = RiskAnalysis(subset).pca(num_components=5)
                return subset.index[-1], np.round(np.real(portfolio_components[pc].to_numpy()), 6)
            except Exception:
                return None

        results_eigen = {}
        with ThreadPoolExecutor(max_workers=cpu_count()) as executor:
            futures = [executor.submit(worker, *task) for task in tasks]
            for future in as_completed(futures):
                out = future.result()
                if out is not None:
                    results_eigen[out[0]] = out[1]
                loading_bar_market.value += 1

        if not results_eigen:
            with market_factor_output:
                print("⚠️ No valid Eigen values computed.")
            return

        # Weight vectors come back in returns_to_use column order -> label them.
        weights = pd.DataFrame(results_eigen, index=asset_cols).T.sort_index()
        quantities_eigen = rebalanced_dynamic_quantities(range_prices, weights)

        market_index = (quantities_eigen * range_prices).sum(axis=1).to_frame('Market Index')
        perf_index_eigen = market_index.pct_change(fill_method=None)
        if not performance_ex_post.empty:
            perf_index_eigen = pd.concat([perf_index_eigen, performance_ex_post], axis=1)
        elif not global_returns.empty:
            perf_index_eigen = pd.concat([perf_index_eigen, global_returns], axis=1)

        show_market_graph(None)

    refresh_market_dates_button = widgets.Button(description='Refresh')
    market_factors_button.on_click(get_market_factors)
    refresh_market_dates_button.on_click(show_market_graph)
    market_factors_ui = widgets.VBox([widgets.HBox([start_date_market_risk, end_date_market_risk, market_factors_button, refresh_market_dates_button]),
                                      widgets.VBox([frequency_eigen, selected_pca_market, market_vol_window, market_factor_output])])

    # =========================================================================
    # CURRENT PORTFOLIO TAB -- ex-post performance from the Excel files
    # =========================================================================
    start_date_perf_ex_post = widgets.DatePicker(value=datetime.date.today(), layout=widgets.Layout(width='350px'))
    end_date_perf_ex_post = widgets.DatePicker(value=datetime.date.today(), layout=widgets.Layout(width='350px'))
    ex_post_perf = widgets.Output()
    ex_post_calendar = widgets.Output()
    performance_output = widgets.Output()
    fund_ex_post = widgets.Dropdown(options=[], description='Select Fund:', style={'description_width': '150px'})
    benchmark_ex_post = widgets.Dropdown(options=[], description='Select Benchmark:', style={'description_width': '150px'})
    frequency_graph_ex_post = widgets.Dropdown(options=['Year', 'Month'], value='Year', description='Select Frequency:', style={'description_width': '150px'})
    calendar_button_ex_post = widgets.Button(description='Update', button_style='info')

    def positions_in_base():
        """Positions.xlsx values converted from POSITIONS_CURRENCY to BASE_CURRENCY."""
        if positions_raw.empty or POSITIONS_CURRENCY == BASE_CURRENCY:
            return positions_raw.copy()
        fx = fx_rates([POSITIONS_CURRENCY], positions_raw.index[0].date())
        rate = ffill_align(fx[[POSITIONS_CURRENCY]], positions_raw.index)[POSITIONS_CURRENCY].bfill().fillna(1.0)
        return positions_raw.mul(rate, axis=0)

    def load_excel_data(_=None):
        """(Re)load the four Excel files and rebuild the trade ledger."""
        global positions, quantities_holding, pnl_history, daily_pnl
        global current_holdings, trades, book_cost, realized_pnl, trade_net_quantities, ledger, positions_raw
        with ex_post_perf:
            ex_post_perf.clear_output(wait=True)
            pos = load_excel(positions_path)
            qty = load_excel(quantities_path)
            if pos is None or pos.empty:
                print("❌ Positions file could not be loaded.")
            else:
                pos = pos.drop(columns='Total', errors='ignore')
                pos['Total'] = pos.sum(axis=1)
                positions_raw = pos
                positions = positions_in_base()
                start_date_perf_ex_post.value = positions.index[0].date()
                print(f"✅ Positions {positions.index[0].date()} → {positions.index[-1].date()}")
            quantities_holding = qty if qty is not None else pd.DataFrame()

            current_holdings = read_current_portfolio(current_portfolio_path)
            if not current_holdings.empty:
                print(f"✅ Current portfolio: {len(current_holdings)} lines")

            trades = read_trade_history(trades_path)
            ledger = build_ledger(trades)
            book_cost = ledger['book_cost']
            realized_pnl = ledger['realized_pnl']
            trade_net_quantities = ledger['net_quantities']
            if not trades.empty:
                print(f"✅ {len(trades)} trades {trades['Date'].min().date()} → {trades['Date'].max().date()}")

            # rebuilt with prices on 'Get P&L'
            pnl_history = pd.DataFrame()
            daily_pnl = pd.DataFrame()
        get_holdings(None)
        refresh_trajectory_choices()

    reload_excel_button.on_click(load_excel_data)

    def apply_saved_histories(quantities_df, positions_df):
        """Use the files just written directly -- re-downloading a GitHub file
        right after a push can return the cached old version for a few minutes."""
        global positions, positions_raw, quantities_holding
        pos = positions_df.drop(columns='Total', errors='ignore')
        pos['Total'] = pos.sum(axis=1)
        positions_raw = pos
        positions = positions_in_base()
        quantities_holding = quantities_df.copy()

    def on_update_files(_=None):
        """Save today's holdings (and fill missed days) into the Quantities /
        Positions files the app is using -- written locally, or pushed to the
        same file on GitHub when the path is a GitHub link."""
        with update_files_output:
            update_files_output.clear_output(wait=True)
            print("Saving today's holdings...")
            try:
                links = [p for p in (quantities_path, positions_path)
                         if str(p).lower().startswith(('http://', 'https://'))]
                if links and git is None:
                    raise ValueError("the Quantities / Positions paths are links -- pass git=your_git to push them.")
                units_today = read_current_portfolio(current_portfolio_path)
                if units_today.empty:
                    raise ValueError(f"nothing read from {current_portfolio_path}")
                q_hist = load_excel(quantities_path)
                p_hist = load_excel(positions_path)
                if (q_hist is None and not quantities_holding.empty) or (p_hist is None and not positions_raw.empty):
                    # don't overwrite a history we just couldn't download
                    raise ValueError("couldn't re-read the Quantities / Positions files (see above).")
                q_new, p_new, summary, info = update_files(units_today, q_hist, p_hist,
                                                           base=POSITIONS_CURRENCY, trades=trades)
            except Exception as e:
                print(f"❌ Files not updated: {e}")
                return
            for df, path in ((q_new, quantities_path), (p_new, positions_path)):
                if path in links:
                    file_name = os.path.splitext(os.path.basename(str(path).split('?')[0]))[0]
                    git.push_or_update_file(df, file_name, folder=git_folder)
                else:
                    df.to_excel(path)
                    print(f"💾 Saved to {path}")
            for line in describe_update(summary, info, POSITIONS_CURRENCY):
                print(line)
        apply_saved_histories(q_new, p_new)
        get_holdings(None)
        if not historical_ptf.empty:
            get_ex_post_returns(None)         # refresh the P&L / return charts

    update_files_button.on_click(on_update_files)

    def build_performance_ex_post():
        """Daily returns of every portfolio over the P&L tab's dates, built from
        exactly the same weights and returns as Return Analysis (yesterday's
        weights x today's simple returns, base currency, trading days). The
        Cumulative Return chart therefore matches the return contributions by
        construction, instead of depending on the Strategy tab's own dates and
        rebalancing frequency (global_returns)."""
        if historical_ptf.empty:
            return pd.DataFrame()
        rng = parse_range(start_date_perf_ex_post, end_date_perf_ex_post)
        if rng is None:
            return historical_ptf[['Historical Portfolio']].copy()
        weights_by_name, returns, _ = trajectory_inputs(*rng)
        series = {}
        for name, w in weights_by_name.items():
            held = w.shift(1)
            r = (held * returns.reindex(held.index)).sum(axis=1, min_count=1)
            series[name] = r.where(held.notna().any(axis=1))
        out = pd.DataFrame(series).dropna(axis=1, how='all')
        if 'Historical Portfolio' not in out.columns:
            # no prices for the shared returns (e.g. offline): fall back to the
            # historical series built in 'Get P&L'
            out = pd.concat([historical_ptf[['Historical Portfolio']], out], axis=1)
        cols = ['Historical Portfolio'] + [c for c in out.columns if c != 'Historical Portfolio']
        return out[cols].sort_index()

    def refresh_ex_post_options(perf):
        options = list(perf.columns)
        set_choices(fund_ex_post, options, 'Historical Portfolio')
        others = [o for o in options if o != 'Historical Portfolio']
        set_choices(benchmark_ex_post, options, ['Fund', *others])

    def show_graph_ex_post(_):
        rng = parse_range(start_date_perf_ex_post, end_date_perf_ex_post)
        with ex_post_calendar:
            ex_post_calendar.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            perf = build_performance_ex_post()
            if perf.empty:
                print("⚠️ Click 'Get P&L' first.")
                return
            if perf.shape[1] < 2:
                print("⚠️ Run an optimization to get something to compare against.")
                return
            if fund_ex_post.value == benchmark_ex_post.value:
                print("⚠️ Benchmark and Fund must be different.")
                return
            cumulative = perf.loc[rng[0]:rng[1]].copy()
            cumulative.iloc[0] = 0
            cumulative = (1 + cumulative).cumprod() * 100
            graphs = get_calendar_graph(cumulative, freq=frequency_graph_ex_post.value,
                                        benchmark=benchmark_ex_post.value, fund=fund_ex_post.value)
            keys = list(graphs.keys())
            left, right = widgets.Output(), widgets.Output()
            with left:
                graphs[keys[0]].show()
                graphs[keys[2]].show()
            with right:
                graphs[keys[1]].show()
                graphs[keys[3]].show()
            display(widgets.HBox([left, right]))

    calendar_button_ex_post.on_click(show_graph_ex_post)

    def show_performance_chart(_):
        global performance_ex_post
        rng = parse_range(start_date_perf_ex_post, end_date_perf_ex_post)
        with performance_output:
            performance_output.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if historical_ptf.empty:
                print("⚠️ Click 'Get P&L' first.")
                return
        start_ts, end_ts = rng

        performance_ex_post = build_performance_ex_post()
        fund_col, bench_col = fund_ex_post.value, benchmark_ex_post.value

        # Same simple returns (base currency, trading days) for both portfolios,
        # yesterday's weights earning today's return.
        with performance_output:
            weights_by_name, returns, msgs = trajectory_inputs(start_ts, end_ts)
            for msg in msgs:
                print(msg)
        fund_w = resolve_weight_series(weights_by_name, fund_col)
        bench_w = resolve_weight_series(weights_by_name, bench_col)
        if fund_w is None or bench_w is None:
            with performance_output:
                print("⚠️ No weight history for the selected fund/benchmark -- run the optimization first.")
            return
        held_f = fund_w.shift(1).dropna(how='all')
        held_b = bench_w.shift(1).dropna(how='all')
        # both measured from the same first day, otherwise the gap isn't comparable
        idx = held_f.index.intersection(held_b.index)
        if len(idx) < 2:
            with performance_output:
                print(f"⚠️ {fund_col} and {bench_col} have no common history in this date range.")
            return
        r = returns.loc[idx]
        held_f, held_b = held_f.loc[idx], held_b.loc[idx]

        performance_contrib = return_contributions(held_f, r)
        performance_contrib_bench = return_contributions(held_b, r)
        excess_returns = performance_contrib - performance_contrib_bench
        excess_returns = excess_returns.loc[:, (excess_returns.abs() > 1e-12).any()]

        contribution_to_drawdown = drawdown_contribution((held_f * r).fillna(0.0))
        value = 1 + performance_contrib['Total Return']
        drawdown = pd.concat([contribution_to_drawdown, (value / value.cummax() - 1).rename(fund_col)], axis=1)

        with performance_output:
            gap = excess_returns['Total Return'].iloc[-1]
            print(f"From {idx[0].date()}: {fund_col} {performance_contrib['Total Return'].iloc[-1]:+.2%}, "
                  f"{bench_col} {performance_contrib_bench['Total Return'].iloc[-1]:+.2%}, active {gap:+.2%}")

        with performance_output:
            performance_output.clear_output(wait=True)
            left, right = widgets.Output(), widgets.Output()
            kw = dict(pct_axis=True, width=800, height=400, xaxis_title=None, yaxis_title=None)
            with left:
                only_show(dark_layout(px.line(performance_contrib, title=f'{fund_col} Return Contribution', render_mode='svg'), **kw), ['Total Return']).show()
                only_show(dark_layout(px.line(performance_contrib_bench, title=f'{bench_col} Return Contribution', render_mode='svg'), **kw), ['Total Return']).show()
            with right:
                only_show(dark_layout(px.line(drawdown, title='Drawdown', render_mode='svg'), **kw), [fund_col]).show()
                only_show(dark_layout(px.line(excess_returns, title=f'Active Return Contribution ({fund_col} vs {bench_col})', render_mode='svg'), **kw), ['Total Return']).show()
            display(widgets.HBox([left, right]))

    def update_ex_post_chart(_):
        global performance_ex_post
        rng = parse_range(start_date_perf_ex_post, end_date_perf_ex_post)
        with ex_post_perf:
            ex_post_perf.clear_output()
            if rng is None:
                print("⚠️ Error with date range.")
                return
            if historical_ptf.empty:
                print("⚠️ Click 'Get P&L' first.")
                return
        start_ts, end_ts = rng

        performance_ex_post = build_performance_ex_post()
        cumulative = performance_ex_post.loc[start_ts:end_ts].copy()
        cumulative.iloc[0] = 0
        cumulative = (1 + cumulative).cumprod() * 100
        selected_positions = positions.loc[start_ts:end_ts]

        with ex_post_perf:
            ex_post_perf.clear_output(wait=True)
            left, right = widgets.Output(), widgets.Output()
            kw = dict(width=800, height=400, xaxis_title=None, yaxis_title=None)
            with left:
                only_show(dark_layout(px.line(selected_positions, title='Portfolio Value', render_mode='svg'), **kw), ['Total']).show()
            with right:
                only_show(dark_layout(px.line(cumulative, title='Cumulative Return', render_mode='svg'), **kw),
                          ['Historical Portfolio', 'Fund', BENCHMARK_LABEL]).show()

            pnl_slice = pnl_history.loc[start_ts:end_ts] if not pnl_history.empty else pd.DataFrame()
            if pnl_slice.empty:
                display(widgets.HBox([left, right]))
                print("ℹ️ No P&L file for this range -- P&L charts skipped.")
                return

            pnl_contribution = pnl_history.diff().loc[start_ts:end_ts]
            selected_daily = daily_pnl.loc[start_ts:end_ts, 'Total'].copy()
            selected_daily.iloc[0] = 0
            selected_history = pd.concat([selected_daily.cumsum(), pnl_slice['Total']], axis=1)
            selected_history.columns = ['Cumulative P&L', 'Total P&L']
            cumulative_pnl_contrib = pd.concat(
                [pnl_contribution.drop(columns='Total').cumsum(), selected_history], axis=1)

            with left:
                only_show(dark_layout(px.line(cumulative_pnl_contrib, title='Cumulative P&L Contribution', render_mode='svg'), **kw),
                          ['Cumulative P&L']).show()
            with right:
                fig8 = px.bar(pnl_contribution.drop(columns='Total'), title="Daily P&L", barmode="relative")
                dark_layout(fig8, showlegend=True, **kw).show()
            display(widgets.HBox([left, right]))
            display(display_scrollable_df(pnl_contribution))

    def get_ex_post_returns(_):
        """Realized returns (Excel weights x prices) and P&L history (trade ledger)."""
        global historical_ptf, performance_ex_post, pnl_history, daily_pnl

        if positions.empty:
            load_excel_data(None)
        with ex_post_perf:
            ex_post_perf.clear_output()
            if positions.empty:
                print("❌ Positions not available.")
                return
            weights = weights_ex_post_frame()
            start_hold = weights.index[0].date()
            prices_hold = get_price_threading(weights.columns, start_hold)      # quote currency
            quote_ccy = quote_currencies(weights.columns, trades)
            trade_ccy = ticker_currencies(trades, weights.columns, quote=quote_ccy)
            fx = fx_rates(set(quote_ccy.values) | set(trade_ccy.values), start_hold)
            # Offline / missing tickers: fall back to prices implied by the
            # Excel history (value / units, already in BASE_CURRENCY).
            implied = implied_prices()
            gaps = [c for c in weights.columns if c not in prices_hold.columns and c in implied.columns]
            if not prices_hold.empty:
                prices_base = to_base_currency(prices_hold, quote_ccy, fx)
            else:
                prices_base = pd.DataFrame()
            if gaps:
                bdays = prices_base.index if not prices_base.empty else pd.bdate_range(implied.index[0], implied.index[-1])
                fill = ffill_align(implied[gaps], bdays)
                prices_base = pd.concat([prices_base, fill], axis=1) if not prices_base.empty else fill
                print(f"ℹ️ Prices implied from Positions/Quantities for: {', '.join(gaps)}")
            if prices_base.empty:
                print("❌ Could not load prices for the held assets.")
                return

        # Weights are in BASE_CURRENCY, so returns are computed on base prices.
        simple_returns = prices_base.pct_change(fill_method=None)
        common = simple_returns.columns.intersection(weights.columns)
        missing = sorted(set(weights.columns) - set(common))
        # Excel may contain dates the price feed doesn't (weekends); carry the
        # last known weights forward and use yesterday's weights for today's return.
        w = ffill_align(weights[common], simple_returns.index).shift(1)
        valid = w.notna().any(axis=1)
        contrib = (simple_returns[common] * w)[valid]

        historical_ptf = contrib.copy()
        historical_ptf['Historical Portfolio'] = contrib.sum(axis=1)

        if not trades.empty:
            # P&L from the trades only: units, cost, realized and fees all come
            # from the ledger; the Excel snapshots are not used here.
            last_day = max(prices_base.index.max(), ledger['units_daily'].index.max())
            cal = pd.date_range(ledger['units_daily'].index.min(), last_day, freq='D')
            trade_units = asof_frame(ledger['units_daily'], cal, list(ledger['units_daily'].columns), fill=0.0)
            trade_ccy = ticker_currencies(trades, trade_units.columns, quote=quote_currencies(trade_units.columns, trades))
            fx = fx_rates(set(fx.columns) | set(trade_ccy.values), start_hold)
            missing_px = [t for t in trade_units.columns if t not in prices_base.columns]
            if missing_px:
                extra = get_price_threading(missing_px, cal[0].date())
                if not extra.empty:
                    prices_base = pd.concat([prices_base, to_base_currency(extra, quote_currencies(extra.columns, trades), fx)], axis=1)
            pnl_history = pnl_history_from_ledger(trade_units, prices_base, ledger['book_cost_daily'],
                                                  ledger['realized_daily'], fx=fx, trade_ccy=trade_ccy,
                                                  fees=ledger['fees_daily'])
            daily_pnl = pnl_history['Total'].diff().to_frame('Total')
            with ex_post_perf:
                for msg in pnl_coverage_report(trade_units, prices_base, ledger):
                    print(msg)
        else:
            pnl_history, daily_pnl = pd.DataFrame(), pd.DataFrame()
            with ex_post_perf:
                print("⚠️ P&L not computed: no trades loaded (Trade History.xlsx)")

        performance_ex_post = build_performance_ex_post()
        refresh_ex_post_options(performance_ex_post)
        update_ex_post_chart(None)
        with ex_post_perf:
            if missing:
                print(f"⚠️ No prices for: {', '.join(missing)} (treated as 0 return)")
            if pnl_history.empty:
                print("ℹ️ P&L needs Trade History.xlsx and Quantities.xlsx.")
        show_graph_ex_post(None)
        show_performance_chart(None)

    ex_post_button = widgets.Button(description='Get P&L', button_style='info')
    ex_post_button.on_click(get_ex_post_returns)
    refresh_ex_post_button = widgets.Button(description='Refresh')
    refresh_ex_post_button.on_click(update_ex_post_chart)
    refresh_performance_analysis_button = widgets.Button(description='Refresh')
    refresh_performance_analysis_button.on_click(show_performance_chart)

    ex_post_ui = widgets.VBox([widgets.HBox([start_date_perf_ex_post, end_date_perf_ex_post, ex_post_button,
                                             refresh_ex_post_button, update_files_button]),
                               update_files_output, ex_post_perf])
    calendar_ui_ex_post = widgets.VBox([widgets.HBox([frequency_graph_ex_post, fund_ex_post, benchmark_ex_post, calendar_button_ex_post]), ex_post_calendar])
    performance_analysis_ui = widgets.VBox([widgets.HBox([start_date_perf_ex_post, end_date_perf_ex_post, ex_post_button, refresh_performance_analysis_button]),
                                            widgets.HBox([fund_ex_post, benchmark_ex_post]), performance_output])

    # =========================================================================
    # RISK ANALYSIS TAB -- ex-ante vol & tracking-error trajectories
    # =========================================================================
    results_vol = pd.DataFrame()
    results_tracking_error = pd.DataFrame()
    spread_weights = {}

    risk_trajectory_output = widgets.Output()
    tracking_error_trajectory_output = widgets.Output()
    selected_fund_to_decompose = widgets.Dropdown(options=[], description='Fund:')    # Risk Trajectory
    selected_fund_te = widgets.Dropdown(options=[], description='Fund:')              # Tracking Error Trajectory
    selected_bench_risk = widgets.Dropdown(options=[], description='Bench:')
    window_risk = widgets.IntText(value=252, description='Window:')
    window_te = widgets.IntText(value=252, description='Window:')
    risk_trajectory_button = widgets.Button(description="Get Ex Ante Vol", button_style="success")
    tracking_error_trajectory_button = widgets.Button(description="Get Ex Ante TE", button_style="success")
    risk_trajectory_refresh_button = widgets.Button(description="Refresh")
    tracking_error_refresh_button = widgets.Button(description="Refresh")

    def trajectory_precheck(out, window):
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        with out:
            out.clear_output(wait=True)
            if rng is None:
                print("⚠️ Invalid start or end date.")
                return None
            if dataframe.empty or returns_to_use.empty or grid.data.empty:
                print("⚠️ Please compute optimization results first.")
                return None
            if positions.empty:
                print("⚠️ Positions not loaded -- click 'Reload Excel'.")
                return None
            if len(returns_to_use.loc[rng[0]:rng[1]]) < window:
                print("⚠️ Date range is shorter than rolling window.")
                return None
        return rng

    def plot_decomposition(out, total_df, title, weights, returns, window, total_name, highlight=()):
        start_ts, end_ts = parse_range(start_date_perf_risk, end_date_perf_risk)
        w = weights.loc[start_ts:end_ts]
        r = returns.reindex(w.index)
        contribution = get_ex_ante_vol_contribution(w, r, window)
        correlation = get_correlation_contribution(w, r, window)
        idiosyncratic = get_idiosyncratic_contribution(w, r, window)
        with out:
            out.clear_output(wait=True)
            left, right = widgets.Output(), widgets.Output()
            with left:
                fig = px.line(total_df.loc[start_ts:end_ts], title=title, width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig, pct_axis=True), list(highlight) or ["Historical Portfolio", "Fund"]).show()
                fig4 = px.line(idiosyncratic, title='Idiosyncratic Contribution', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig4, pct_axis=True), ["Total Idiosyncratic Vol"]).show()
            with right:
                fig2 = px.line(contribution, title=f'{total_name} Contribution', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig2, pct_axis=True), ["Total Vol"]).show()
                fig3 = px.line(correlation, title='Correlation Contribution', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig3, pct_axis=True), ["Total Correlation"]).show()
            display(widgets.HBox([left, right]))

    def trajectory_inputs(start_ts, end_ts):
        """Every portfolio's weights on ONE calendar with ONE return table.

        * Returns: simple daily returns, in BASE_CURRENCY when conversion is on,
          for every universe, model and held ticker, on trading days only.
        * Weights: each series (model allocations, Fund/Core/Overlay, and the
          Historical Portfolio from Positions.xlsx) carried forward onto those
          trading days -- Positions.xlsx has weekend rows, which previously put
          NaN returns into the rolling window.
        * Columns: the same for every portfolio (0 where not held), so model
          vs. historical spreads keep assets held on only one side.
        Returns (weights_by_name, returns, messages).
        """
        series = build_weight_series_dict(start_ts, end_ts)
        w_hist = weights_ex_post_frame()
        tick = set(dataframe.columns) | set(quantities.columns) | set(w_hist.columns)
        first = min(start_ts, w_hist.index[0]) if not w_hist.empty else start_ts
        px_, _ = prices_to_base(get_price_threading(tick, first.date()), first.date())
        returns = px_.pct_change(fill_method=None).loc[start_ts:end_ts]
        returns = returns.loc[returns.notna().any(axis=1)]
        if not w_hist.empty:
            series['Historical Portfolio'] = w_hist.loc[:end_ts]

        weights, msgs = {}, []
        for key, w in series.items():
            a = ffill_align(w, returns.index).dropna(how='all')      # rows before the portfolio existed
            unpriced = [c for c in a.columns if c not in returns.columns and (a[c].abs() > 1e-9).any()]
            if unpriced:
                msgs.append(f"⚠️ {key}: no prices for {', '.join(unpriced)} -- left out of its risk.")
            weights[key] = a.reindex(columns=returns.columns).fillna(0.0)
        return weights, returns.fillna(0.0), msgs

    def as_series(result, name):
        """get_ex_ante_vol may return a Series or a one-column DataFrame."""
        if isinstance(result, pd.DataFrame):
            if result.shape[1] == 1:
                result = result.iloc[:, 0]
            else:
                col = next((c for c in ('Total Vol', 'Portfolio', 'Total') if c in result.columns), result.columns[-1])
                result = result[col]
        return pd.Series(result, name=name)

    def run_trajectories(weights_by_name, returns, window, out):
        results, short = {}, []
        loading_bar_risk.value = 0
        loading_bar_risk.max = max(len(weights_by_name), 1)
        for name, w in weights_by_name.items():
            if len(w) < window:
                short.append(f"{name} ({len(w)} days)")
            else:
                results[name] = as_series(get_ex_ante_vol(w, returns.loc[w.index], window), name)
            loading_bar_risk.value += 1
        if short:
            with out:
                print(f"⚠️ Fewer days than the {window}-day window, skipped: {', '.join(short)}. "
                      "Lower the window or widen the dates.")
        return pd.DataFrame(results)

    risk_state = {}    # weights / returns of the last 'Get Ex Ante Vol' (own copy, not shared with VaR / TE)

    def show_risk_graph(_):
        with risk_trajectory_output:
            risk_trajectory_output.clear_output(wait=True)
            if results_vol.empty or 'series' not in risk_state:
                print("⚠️ Click 'Get Ex Ante Vol' first.")
                return
            if parse_range(start_date_perf_risk, end_date_perf_risk) is None:
                print("⚠️ Invalid start or end date.")
                return
        name = selected_fund_to_decompose.value
        if name not in risk_state['series']:
            with risk_trajectory_output:
                print(f"⚠️ No weights for '{name}'.")
            return
        plot_decomposition(risk_trajectory_output, results_vol, 'Ex Ante Volatility',
                           risk_state['series'][name], risk_state['returns'], window_risk.value, 'Volatility',
                           highlight=[name])

    def get_risk_trajectory(_):
        global results_vol
        rng = trajectory_precheck(risk_trajectory_output, window_risk.value)
        if rng is None:
            return
        with risk_trajectory_output:
            display(loading_bar_risk)
            risk_series, risk_returns, msgs = trajectory_inputs(*rng)
            risk_state.update(series=risk_series, returns=risk_returns)
            for msg in msgs:
                print(msg)
        results_vol = run_trajectories(risk_series, risk_returns, window_risk.value, risk_trajectory_output)
        if results_vol.empty:
            with risk_trajectory_output:
                print("⚠️ Risk results are empty.")
            return
        cols = list(results_vol.columns)
        set_choices(selected_fund_to_decompose, cols, 'Historical Portfolio')
        show_risk_graph(None)

    te_state = {}      # inputs of the last 'Get Ex Ante TE' + benchmark/window the results are for

    def compute_tracking_errors(bench_name, window, out):
        """Active weights vs `bench_name` and every portfolio's ex-ante TE
        trajectory (the general chart) -- run by 'Get Ex Ante TE' only."""
        global results_tracking_error, spread_weights
        series, returns = te_state['series'], te_state['returns']
        if bench_name not in series:
            fallback = 'Fund' if 'Fund' in series else next(iter(series))
            with out:
                print(f"⚠️ Benchmark '{bench_name}' has no weight history -- using '{fallback}'.")
            bench_name = fallback
        bench = series[bench_name]
        # active weights on the dates both portfolios exist; the benchmark vs
        # itself is identically 0, so it's left out
        spread_weights = {}
        for key, w in series.items():
            if key == bench_name:
                continue
            idx = w.index.intersection(bench.index)
            spread_weights[key] = w.loc[idx] - bench.loc[idx]
        with out:
            display(loading_bar_risk)
        results_tracking_error = run_trajectories(spread_weights, returns, window, out)
        te_state.update(bench=bench_name, window=window)
        return bench_name

    def show_tracking_error_graph(_):
        """Refresh: the 3 breakdown charts follow the Fund / Bench dropdowns
        (active weights rebuilt from the loaded data, no download). The general
        chart keeps the last 'Get Ex Ante TE' result -- click that button to
        recompute it against another benchmark or window."""
        with tracking_error_trajectory_output:
            tracking_error_trajectory_output.clear_output(wait=True)
            if 'series' not in te_state or results_tracking_error.empty:
                print("⚠️ Click 'Get Ex Ante TE' first.")
                return
            if parse_range(start_date_perf_risk, end_date_perf_risk) is None:
                print("⚠️ Invalid start or end date.")
                return
            series = te_state['series']
            name, bench_name = selected_fund_te.value, selected_bench_risk.value
            if name not in series or bench_name not in series:
                print("⚠️ Select a fund and a benchmark.")
                return
            if name == bench_name:
                print("⚠️ Fund and benchmark are the same -- the tracking error is 0.")
                return
        fund_w, bench_w = series[name], series[bench_name]
        idx = fund_w.index.intersection(bench_w.index)
        active = fund_w.loc[idx] - bench_w.loc[idx]
        plot_decomposition(tracking_error_trajectory_output, results_tracking_error,
                           f"Ex Ante Tracking Error vs {te_state['bench']}",
                           active, te_state['returns'], window_te.value, 'Tracking Error',
                           highlight=[name] if name in results_tracking_error.columns else ())

    def refresh_trajectory_choices():
        """Names the risk / VaR / TE trajectories will have, so every Fund and
        Bench dropdown can be picked before the first run (same keys as
        build_weight_series_dict + Historical Portfolio)."""
        names = ['Historical Portfolio'] if not positions.empty else []
        if grid.data is not None and not grid.data.empty:
            for key in grid.data.index:
                names += ['Rebalanced ' + key, 'Buy and Hold ' + key]
        names += [n for n, q in (('Fund', quantities), ('Core', quantities_core), ('Overlay', quantities_overlay))
                  if not q.empty]
        if BENCHMARK_TICKER in dataframe.columns:
            names.append(BENCHMARK_LABEL)
        if not names:
            return
        # Fund dropdowns are filled right away (not only after the first run);
        # the user's current choice is kept.
        set_choices(selected_bench_risk, names, ['Fund', BENCHMARK_LABEL])
        set_choices(selected_fund_te, names,
                [n for n in ('Historical Portfolio', *names) if n != selected_bench_risk.value])
        others = [n for n in names if n != selected_bench_risk.value]
        if selected_fund_te.value == selected_bench_risk.value and others:
            setattr(selected_fund_te, 'value', others[0])     # never start on fund == bench
        set_choices(selected_fund_to_decompose, names, 'Historical Portfolio')
        set_choices(selected_fund_to_decompose_var,
                [n for n in names if n != BENCHMARK_LABEL], ['Fund', 'Historical Portfolio'])

    def get_tracking_error_trajectory(_):
        rng = trajectory_precheck(tracking_error_trajectory_output, window_te.value)
        if rng is None:
            return
        with tracking_error_trajectory_output:
            display(loading_bar_risk)
            series, returns, msgs = trajectory_inputs(*rng)
            for msg in msgs:
                print(msg)
        te_state.clear()
        te_state.update(series=series, returns=returns)
        bench_name = compute_tracking_errors(selected_bench_risk.value, window_te.value,
                                             tracking_error_trajectory_output)
        if results_tracking_error.empty:
            with tracking_error_trajectory_output:
                print("⚠️ Risk results are empty.")
            return
        set_choices(selected_bench_risk, list(series), bench_name)
        set_choices(selected_fund_te, list(series), [n for n in ('Historical Portfolio', *series) if n != bench_name])
        if selected_fund_te.value == bench_name:            # don't start on fund == benchmark
            setattr(selected_fund_te, 'value', next(n for n in ('Historical Portfolio', *series) if n != bench_name))
        show_tracking_error_graph(None)

    risk_trajectory_button.on_click(get_risk_trajectory)
    risk_trajectory_refresh_button.on_click(show_risk_graph)
    tracking_error_trajectory_button.on_click(get_tracking_error_trajectory)
    tracking_error_refresh_button.on_click(show_tracking_error_graph)
    risk_exposure_ui = widgets.VBox([widgets.HBox([start_date_perf_risk, end_date_perf_risk, risk_trajectory_button, risk_trajectory_refresh_button]),
                                     selected_fund_to_decompose, window_risk, risk_trajectory_output])
    tracking_error_exposure_ui = widgets.VBox([widgets.HBox([start_date_perf_risk, end_date_perf_risk, tracking_error_trajectory_button, tracking_error_refresh_button]),
                                               selected_fund_te, selected_bench_risk, window_te, tracking_error_trajectory_output])

    # =========================================================================
    # CURRENT PORTFOLIO TAB -- rolling beta vs. a chosen benchmark
    # =========================================================================
    beta_output = widgets.Output()
    window_beta = widgets.IntText(value=252, description='Window:', style={'description_width': '150px'})
    get_beta_button = widgets.Button(description='Get Beta', button_style='info')
    refresh_beta_button = widgets.Button(description='Refresh')
    beta_state = {}

    def get_beta_trajectory(_):
        """Expensive step (price download); re-run when dates/fund/bench change."""
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        with beta_output:
            beta_output.clear_output(wait=True)
            if rng is None:
                print("⚠️ Invalid start or end date.")
                return
            if dataframe.empty or returns_to_use.empty or grid.data.empty:
                print("⚠️ Please compute optimization results first.")
                return
            if historical_ptf.empty:
                print("⚠️ Click 'Get P&L' first.")
                return
            if len(returns_to_use.loc[rng[0]:rng[1]]) < window_beta.value:
                print("⚠️ Date range is shorter than rolling window.")
                return
            weights_ex_post = weights_ex_post_frame()
            beta_state['current_underlying_returns'] = fetch_underlying_returns(weights_ex_post)
            beta_state['performance_ex_post'] = build_performance_ex_post()
        show_beta_graph(None)

    def show_beta_graph(_):
        """Cheap redraw from cached beta_state (no network)."""
        rng = parse_range(start_date_perf_risk, end_date_perf_risk)
        with beta_output:
            beta_output.clear_output(wait=True)
            if rng is None:
                print("⚠️ Invalid start or end date.")
                return
            if not beta_state:
                print("⚠️ Click 'Get Beta' first.")
                return
            start_ts, end_ts = rng
            underlying = beta_state['current_underlying_returns']
            perf = beta_state['performance_ex_post']
            bench_col = benchmark_ex_post.value
            if bench_col not in perf.columns:
                print(f"⚠️ Benchmark '{bench_col}' not available -- click 'Get Beta' again.")
                return

            series_local = build_weight_series_dict(start_ts, end_ts)
            series_local['Historical Portfolio'] = weights_ex_post_frame().loc[start_ts:end_ts]
            selected_weights = resolve_weight_series(series_local, fund_ex_post.value)
            if selected_weights is None:
                print(f"⚠️ '{fund_ex_post.value}' has no weight series available.")
                return

            beta_rolling = compute_rolling_per_asset_betas(underlying, perf[bench_col], window_beta.value)
            common_cols = beta_rolling.columns.intersection(selected_weights.columns)
            weights_aligned = ffill_align(selected_weights[common_cols], beta_rolling.index).fillna(0.0)
            weighted_beta_contrib = beta_rolling[common_cols] * weights_aligned
            weighted_beta_contrib['Total Beta'] = weighted_beta_contrib.sum(axis=1, min_count=1)
            fund_name = fund_ex_post.value
            if weighted_beta_contrib['Total Beta'].loc[start_ts:end_ts].notna().sum() == 0:
                overlap = perf[bench_col].dropna().index.intersection(weights_aligned.index)
                print(f"⚠️ No beta for {fund_name} vs {bench_col}: only {len(overlap)} common days of history, "
                      f"the window is {window_beta.value}. Lower the window or widen the dates.")
                return

            perf_beta = compute_rolling_per_asset_betas(perf, perf[bench_col], window_beta.value)
            # show the fund's largest holdings by default (not a ticker it may not hold)
            last_w = weights_aligned.loc[:end_ts].iloc[-1].abs().sort_values(ascending=False)
            top_holdings = list(last_w[last_w > 0].index[:5])

            left, right = widgets.Output(), widgets.Output()
            with left:
                fig = px.line(beta_rolling[common_cols].loc[start_ts:end_ts], title=f'Rolling Asset Beta vs {bench_col}',
                              width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig), top_holdings).show()
                fig2 = px.line(weights_aligned.loc[start_ts:end_ts], title=f'{fund_name} Weights', width=800, height=400, render_mode='svg')
                dark_layout(fig2, pct_axis=True).show()
            with right:
                fig3 = px.line(perf_beta.loc[start_ts:end_ts], title=f'Portfolio Rolling Beta vs {bench_col}', width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig3), ["Historical Portfolio", "Fund", fund_name]).show()
                fig4 = px.line(weighted_beta_contrib.loc[start_ts:end_ts], title=f'{fund_name} Beta Contribution',
                               width=800, height=400, render_mode='svg')
                only_show(dark_layout(fig4), ["Total Beta"]).show()
            display(widgets.HBox([left, right]))

    get_beta_button.on_click(get_beta_trajectory)
    refresh_beta_button.on_click(show_beta_graph)
    # The beta tab only redraws on 'Get Beta' / 'Refresh' -- changing the fund,
    # benchmark or window does nothing until one of them is clicked.
    beta_ui = widgets.VBox([
        widgets.HBox([start_date_perf_risk, end_date_perf_risk, get_beta_button, refresh_beta_button]),
        fund_ex_post, benchmark_ex_post, window_beta, beta_output])

    # =========================================================================
    # TAB ASSEMBLY
    # =========================================================================
    def tab(children_titles):
        t = widgets.Tab(children=[c for c, _ in children_titles])
        for i, (_, title) in enumerate(children_titles):
            t.set_title(i, title)
        return t

    def wrap(widget):
        out = widgets.Output()
        with out:
            display(widget)
        return out

    strategy_subtabs = tab([(wrap(constraint_ui), 'Strategy'),
                            (wrap(positions_ui), 'Positioning'),
                            (wrap(calendar_perf), 'Strategy Return')])
    pnl_sub_tab = tab([(wrap(ex_post_ui), 'P&L Analysis'),
                       (wrap(performance_analysis_ui), 'Return Analysis'),
                       (wrap(beta_ui), 'Beta Analysis')])
    ex_post_subtabs = tab([(pnl_sub_tab, 'P&L'),
                           (wrap(positions_ui), 'Positioning'),
                           (wrap(calendar_ui_ex_post), 'Calendar Return')])
    risk_subtabs = tab([(wrap(ex_ante_ui), 'Risk Contribution'),
                        (wrap(risk_exposure_ui), 'Risk Trajectory'),
                        (wrap(tracking_error_exposure_ui), 'Tracking Error')])
    var_subtabs = tab([(wrap(var_ui), 'Value at Risk'),
                       (wrap(var_history_ui), 'Value at Risk History')])
    risk_tabs = tab([(risk_subtabs, 'Volatility Analysis'), (var_subtabs, 'Value at Risk')])
    market_risk_subtabs = tab([(wrap(market_ui), 'Market Risk'),
                               (wrap(correlation_ui), 'Correlation'),
                               (wrap(market_factors_ui), 'Market Drivers')])

    main_tabs = tab([(wrap(universe_ui), 'Investment Universe'),
                     (strategy_subtabs, 'Strategy'),
                     (ex_post_subtabs, 'Current Portfolio'),
                     (risk_tabs, 'Risk Analysis'),
                     (market_risk_subtabs, 'Market Risk')])

    load_excel_data(None)
    display(main_tabs)

In [4]:
display_crypto_app(git=git,git_folder='ETF')